In [1]:
import os
import polars as pl
from tqdm.notebook import tqdm
from collections import Counter
pl.Config(tbl_rows=25)

import warnings
warnings.filterwarnings("ignore")


In [2]:
data = pl.read_parquet('../data/EHRSHOT/EHRSHOT_MEDS/data/data.parquet')

In [3]:
labels = pl.read_parquet('../data/EHRSHOT/EHRSHOT_MEDS/labels/guo_los/')

In [4]:
# sid = labels['subject_id'].unique()[4]
# sid

In [5]:
# y = labels.filter(pl.col('subject_id') == sid)
# y

In [6]:
# x = data.filter(pl.col('subject_id') == sid)
# x

In [7]:
# t = y['prediction_time']

In [8]:
# x.filter(pl.col('time')<= t[0])

In [9]:
# pl.read_parquet('../data/EHRSHOT/EHRSHOT_MEDS/labels/guo_icu/labels.parquet')

In [10]:
# pl.read_parquet('../data/EHRSHOT/EHRSHOT_MEDS/labels/lab_hyponatremia/')

In [11]:
# pl.read_parquet('../data/EHRSHOT/EHRSHOT_MEDS/metadata/subject_splits.parquet').columns

In [12]:
import os
import polars as pl
from glob import glob
from tqdm import tqdm




SPLITS_FP = "../data/EHRSHOT/EHRSHOT_MEDS/metadata/subject_splits.parquet"
LABELS_ROOT = "../data/EHRSHOT/EHRSHOT_MEDS/labels/"

# OUTPUT_FP = "./ehrshot_labels_combined.parquet"



splits = pl.read_parquet(SPLITS_FP)

# expected:
# subject_id | split



label_files = glob(os.path.join(LABELS_ROOT, "*", "labels.parquet"))

all_tasks = []



for fp in tqdm(label_files):


    task_name = fp.split(os.sep)[-2]
    labels = pl.read_parquet(fp)
    labels = labels.with_columns(
        pl.lit(task_name).alias("task")
    )
    labels = labels.join(
        splits,
        on="subject_id",
        how="left"
    )
    labels = labels.select([
        "subject_id",
        "split",
        "task",
        "prediction_time",
        "boolean_value",
        "integer_value",
        "float_value",
        "categorical_value",
    ])

    all_tasks.append(labels)

combined = pl.concat(all_tasks, how="vertical")



# combined.write_parquet(OUTPUT_FP)

# print(f"Saved combined labels to: {OUTPUT_FP}")

100%|██████████| 15/15 [00:01<00:00, 14.94it/s]


In [13]:
# combined

In [14]:
import polars as pl


def add_prediction_indices(
    labels: str,
    timelines: str,
#     output_fp: str,
):
    labels = labels.with_columns(
        pl.col("prediction_time").cast(pl.Datetime)
    )

    timelines = timelines.with_columns(
        pl.col("time").cast(pl.Datetime)
    )

    timeline_index = (
        timelines
        .sort(["subject_id", "time"])
        .with_columns(
            pl.arange(0, pl.len()).over("subject_id").alias("event_idx")
        )
        .select(["subject_id", "time", "event_idx"])
    )

    indexed_labels = labels.join_asof(
        timeline_index,
        left_on="prediction_time",
        right_on="time",
        by="subject_id",
        strategy="backward",
    ).rename({
        "event_idx": "prediction_idx"
    })

    indexed_labels = indexed_labels.with_columns(
        pl.lit(0).alias("history_min_idx")
    )

#     indexed_labels.write_parquet(output_fp)
    return indexed_labels




In [15]:
idx = add_prediction_indices(labels=combined,
                             timelines=data)
#     output_fp="ehrshot_labels_indexed.parquet",)

In [ ]:
os.environ['HUGGINGFACE_HUB_TOKEN'] = ""
os.environ["HF_TOKEN"] = ""

hf_token = os.getenv("HF_TOKEN")

CUSTOM_CACHE = "/scratch/sas10092/huggingface_cache"

In [17]:
import femr.models.transformer
import torch
import femr.models.tokenizer
import femr.models.processor
import datetime

import json

In [18]:


dict_fp = "/scratch/sas10092/.cache/huggingface/hub/models--StanfordShahLab--clmbr-t-base/snapshots/c7a5f4db6089525e374c2c90372350db3a043d73/clmbr_v8_original_dictionary.json"

def correct_tokenizer_dict(dict_fp:str):
    with open(dict_fp) as f:
        old_dictionary = json.load(f)
    dictionary = {"age_stats": old_dictionary["age_stats"],
                  "is_hierarchical": old_dictionary.get("is_hierarchical", False),
                  "vocab": old_dictionary["regular"]}
    return dictionary




In [19]:
dictionary = correct_tokenizer_dict(dict_fp)
model_name = "StanfordShahLab/clmbr-t-base"
model = femr.models.transformer.FEMRModel.from_pretrained(model_name)
tokenizer = femr.models.tokenizer.FEMRTokenizer(
    dictionary=dictionary,
    ontology=None)
batch_processor = femr.models.processor.FEMRBatchProcessor(tokenizer)

In [20]:
model

FEMRModel(
  (transformer): FEMRTransformer(
    (in_norm): RMSNorm()
    (out_norm): RMSNorm()
    (embed): Embedding(65536, 768)
    (layers): ModuleList(
      (0-11): 12 x FEMREncoderLayer(
        (norm): RMSNorm()
        (input_proj): Linear(in_features=768, out_features=5376, bias=True)
        (output_proj): Linear(in_features=3840, out_features=768, bias=True)
      )
    )
  )
  (task_model): CLMBRTaskHead(
    (final_layer): Linear(in_features=768, out_features=8192, bias=True)
  )
)

In [21]:
import json
import datetime
import polars as pl
import meds
from collections import defaultdict


def load_clmbr_dictionary(dictionary_fp: str):
    with open(dictionary_fp) as f:
        d = json.load(f)

    vocab = d["vocab"] if "vocab" in d else d["regular"]

    code_set = set()
    text_lookup = set()
    numeric_lookup = defaultdict(list)
    unused_set = set()

    for x in vocab:
        code = x["code_string"]
        typ = x["type"]

        if isinstance(typ, int):
            typ = {0: "code", 1: "text", 2: "numeric", 3: "unused"}.get(typ, "unused")

        if typ == "code":
            code_set.add(code)

        elif typ == "text":
            text_lookup.add((code, str(x["text_string"])))

        elif typ == "numeric":
            numeric_lookup[code].append(
                (float(x["val_start"]), float(x["val_end"]))
            )

        elif typ == "unused":
            unused_set.add(code)

    return {
        "code": code_set,
        "text": text_lookup,
        "numeric": numeric_lookup,
        "unused": unused_set,
    }


def row_is_supported_by_clmbr_dict(
    code,
    numeric_value,
    text_value,
    clmbr_lookup,
):
    if numeric_value is not None:
        if code not in clmbr_lookup["numeric"]:
            return False

        value = float(numeric_value)
        for start, end in clmbr_lookup["numeric"][code]:
            if start <= value < end:
                return True
        return False

    if text_value is not None:
        text = str(text_value)
        return (code, text) in clmbr_lookup["text"]

    return code in clmbr_lookup["code"]


def polars_timeline_to_femr_patient_dictionary_aware(
    df: pl.DataFrame,
    dictionary_fp: str,
    patient_id_col: str = "subject_id",
    time_col: str = "time",
    code_col: str = "code",
    numeric_col: str = "numeric_value",
    text_col: str = "text_value",
    ehrshot_birth_code: str = "SNOMED/3950001",
    add_femr_birth_anchor: bool = True,
    return_stats: bool = True,
):
    if df.height == 0:
        raise ValueError("Input dataframe is empty.")

    clmbr_lookup = load_clmbr_dictionary(dictionary_fp)

    subject_ids = df.get_column(patient_id_col).unique().to_list()
    if len(subject_ids) != 1:
        raise ValueError(f"Expected one subject_id, found {len(subject_ids)}.")

    patient_id = int(subject_ids[0])
    df = df.sort(time_col)

    events = []
    added_birth_anchor = 0

    valid_rows = 0
    invalid_rows = 0
    invalid_examples = []

    for event_time, event_df in df.group_by(time_col, maintain_order=True):
        if isinstance(event_time, tuple):
            event_time = event_time[0]

        if not isinstance(event_time, datetime.datetime):
            event_time = event_time.to_pydatetime()

        measurements = []

        for r in event_df.iter_rows(named=True):
            code = r[code_col]
            numeric_value = r[numeric_col] if numeric_col in event_df.columns else None
            text_value = r[text_col] if text_col in event_df.columns else None

            if code == ehrshot_birth_code and add_femr_birth_anchor:
                measurements.append({"code": meds.birth_code})
                added_birth_anchor += 1

            is_valid = row_is_supported_by_clmbr_dict(
                code=code,
                numeric_value=numeric_value,
                text_value=text_value,
                clmbr_lookup=clmbr_lookup,
            )

            if is_valid:
                valid_rows += 1
            else:
                invalid_rows += 1
                if len(invalid_examples) < 20:
                    invalid_examples.append(
                        {
                            "code": code,
                            "numeric_value": numeric_value,
                            "text_value": text_value,
                            "time": event_time,
                        }
                    )

            m = {"code": code}

            if numeric_value is not None:
                m["numeric_value"] = float(numeric_value)

            if text_value is not None:
                m["text_value"] = str(text_value)

            measurements.append(m)

        if measurements:
            events.append(
                {
                    "time": event_time,
                    "measurements": measurements,
                }
            )

    patient = {
        "patient_id": patient_id,
        "events": events,
    }

    if not return_stats:
        return patient

    stats = {
        "raw_rows": df.height,
        "passed_to_femr_rows": df.height,
        "dictionary_valid_rows": valid_rows,
        "dictionary_invalid_rows": invalid_rows,
        "added_birth_anchor": added_birth_anchor,
        "n_events": len(events),
        "dictionary_invalid_examples": invalid_examples,
    }

    return patient, stats

In [22]:
# from datasets import Dataset
# from tqdm import tqdm
# import torch

# def tensor_to_python(x):
#     if isinstance(x, torch.Tensor):
#         return x.cpu().tolist()
#     if isinstance(x, dict):
#         return {k: tensor_to_python(v) for k, v in x.items()}
#     return x


# records = []

# dictionary_fp = "/scratch/sas10092/.cache/huggingface/hub/models--StanfordShahLab--clmbr-t-base/snapshots/c7a5f4db6089525e374c2c90372350db3a043d73/clmbr_v8_original_dictionary.json"

# for idx in tqdm(data["subject_id"].unique().to_list()):
#     timeline = data.filter(pl.col("subject_id") == idx)

#     patient, stats = polars_timeline_to_femr_patient_dictionary_aware(
#         timeline,
#         dictionary_fp=dictionary_fp,
#     )

#     raw_batch = batch_processor.convert_patient(patient, tensor_type="pt")
#     raw_batch = tensor_to_python(raw_batch)

#     records.append(raw_batch)

# arrow_ds = Dataset.from_list(records)

# arrow_ds.save_to_disk("ehrshot_clmbr_tokenized_arrow")

In [23]:
from datasets import load_from_disk

arrow_ds = load_from_disk('./ehrshot_clmbr_tokenized_arrow/').with_format("torch")
# if isinstance(sample["transformer"]["label_indices"], list):
#     sample["transformer"]["label_indices"] = torch.tensor([], dtype=torch.long)

In [24]:
import polars as pl
import torch
from datasets import load_from_disk
from tqdm import tqdm


from datetime import datetime, timezone

def to_unix_seconds(ts):
    if isinstance(ts, datetime):
        return int(ts.replace(tzinfo=timezone.utc).timestamp())
    return int(ts)


def build_clmbr_label_index(
    labels_df: pl.DataFrame,
    arrow_ds,
    subject_id_col: str = "subject_id",
    prediction_time_col: str = "prediction_time",
):
    arrow_ds = arrow_ds.with_format("torch")

    subject_to_row = {}
    for i in tqdm(range(len(arrow_ds)), desc="Indexing Arrow patients"):
        sid = int(arrow_ds[i][subject_id_col]) if subject_id_col in arrow_ds.column_names else int(arrow_ds[i]["patient_ids"][0])
        subject_to_row[sid] = i

    rows = []

    for r in tqdm(labels_df.iter_rows(named=True), total=labels_df.height, desc="Building CLMBR label index"):
        sid = int(r[subject_id_col])
        pred_time = to_unix_seconds(r[prediction_time_col])

        if sid not in subject_to_row:
            continue

        sample = arrow_ds[subject_to_row[sid]]
        timestamps = sample["transformer"]["timestamps"]

        valid = torch.where(timestamps <= pred_time)[0]
        if len(valid) == 0:
            continue

        prediction_idx = int(valid[-1].item())

        out = dict(r)
        out["arrow_row_idx"] = subject_to_row[sid]
        out["prediction_idx"] = prediction_idx
        out["history_min_idx"] = 0
        out["n_clmbr_tokens"] = int(sample["transformer"]["patient_lengths"][0].item())

        rows.append(out)
    if len(rows) == 0:
        return pl.DataFrame()
    clmbr_idx =  pl.from_dicts(rows, infer_schema_length=None, strict=False)
    clmbr_idx = clmbr_idx.with_row_index("example_id")
    return clmbr_idx

In [25]:
# arrow_ds = load_from_disk("ehrshot_clmbr_tokenized_arrow")
# clmbr_idx = build_clmbr_label_index(combined, arrow_ds)

In [26]:
clmbr_idx = pl.read_parquet('clmbr_idx.parquet')

In [27]:
from datetime import datetime, timezone


def unix_to_dt(x):
    return datetime.fromtimestamp(int(x), tz=timezone.utc).replace(tzinfo=None)


def to_unix_seconds(ts):
    if isinstance(ts, datetime):
        return int(ts.replace(tzinfo=timezone.utc).timestamp())
    return int(ts)


def verify_clmbr_index_row(clmbr_idx, arrow_ds, row_i=0):
    arrow_ds = arrow_ds.with_format("torch")

    row = clmbr_idx.row(row_i, named=True)

    arrow_row_idx = int(row["arrow_row_idx"])
    pred_idx = int(row["prediction_idx"])
    pred_time = row["prediction_time"]
    pred_time_sec = to_unix_seconds(pred_time)

    sample = arrow_ds[arrow_row_idx]
    ts = sample["transformer"]["timestamps"]

    cur_ts = int(ts[pred_idx].item())
    next_ts = int(ts[pred_idx + 1].item()) if pred_idx + 1 < len(ts) else None

    print("subject_id:", row["subject_id"])
    print("task:", row["task"])
    print("prediction_time:", pred_time)
    print("prediction_time_sec:", pred_time_sec)
    print("prediction_idx:", pred_idx)
    print("n_clmbr_tokens:", len(ts))

    print("\nCLMBR timestamp at prediction_idx:")
    print(cur_ts, unix_to_dt(cur_ts))

    print("\nNext CLMBR timestamp:")
    print(None if next_ts is None else (next_ts, unix_to_dt(next_ts)))

    print("\nCheck:")
    print("current <= prediction_time:", cur_ts <= pred_time_sec)
    if next_ts is not None:
        print("next > prediction_time:", next_ts > pred_time_sec)

In [28]:
import os
import json
import torch
import faiss
import bisect
import random
import chromadb
import numpy as np
import polars as pl

from torch import nn

from math import ceil
from pathlib import Path
from collections import defaultdict
from datasets import load_from_disk
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from typing import Dict, Iterable, List, Optional, Any, Union, Literal, Tuple, Callable

In [29]:
class Tokenizer:
    def __init__(
        self,
        codes_parquet_fp: str,
        special_tokens: Optional[Iterable[str]] = None,
        force_special_ids: bool = True,  # pin [PAD]=0 etc.
    ):
        if special_tokens is None:
            special_tokens = ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]


        df_codes = pl.read_parquet(str(codes_parquet_fp), columns=["code"])
        base_codes = df_codes.get_column("code").to_list()
        seen = set()
        unique_codes = []
        for c in base_codes:
            if c not in seen:
                unique_codes.append(c)
                seen.add(c)

        vocab_list: List[str] = []
        special_tokens = list(special_tokens)

        if force_special_ids:
            for tok in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]:
                if tok in special_tokens and tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
                elif tok in special_tokens and tok in seen:

                    vocab_list.append(tok)
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
        else:
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)


        vocab_list.extend(unique_codes)


        self.id2code: List[str] = vocab_list
        self.code2id: Dict[str, int] = {tok: idx for idx, tok in enumerate(vocab_list)}
        self.vocab_size: int = len(self.id2code)

        self.pad_token  = "[PAD]" if "[PAD]" in self.code2id else None
        self.mask_token = "[MASK]" if "[MASK]" in self.code2id else None
        self.cls_token  = "[CLS]" if "[CLS]" in self.code2id else None
        self.unk_token  = "[UNK]" if "[UNK]" in self.code2id else None

        self.pad_id  = self.code2id[self.pad_token]  if self.pad_token  else 0
        self.mask_id = self.code2id[self.mask_token] if self.mask_token else None
        self.cls_id  = self.code2id[self.cls_token]  if self.cls_token  else None
        self.unk_id  = self.code2id[self.unk_token]  if self.unk_token  else None


        type_set = set()
        for tok in self.id2code:
            prefix = tok.split("//", 1)[0]
            type_set.add(prefix)

        types_sorted = sorted(t for t in type_set if t not in ("[PAD]",))
        self.type2id: Dict[str, int] = {"[PAD]": 0}
        next_id = 1
        for sp in ["[MASK]", "[CLS]", "[UNK]"]:
            if sp in type_set:
                self.type2id[sp] = next_id; next_id += 1
        for t in types_sorted:
            if t not in self.type2id:
                self.type2id[t] = next_id
                next_id += 1

        self._code2id_df = pl.DataFrame({"code": self.id2code,
                                         "input_id": list(range(self.vocab_size))}) \
                               .with_columns(pl.col("code").cast(pl.Categorical))
        self._type2id_df = pl.DataFrame({"code_type": list(self.type2id.keys()),
                                         "type_id":   list(self.type2id.values())}) \
                               .with_columns(pl.col("code_type").cast(pl.Categorical))


    def encode(self, codes: Iterable[str]) -> List[int]:
        get = self.code2id.get
        if self.unk_id is not None:
            fallback = self.unk_id
        else:
            fallback = self.pad_id if self.pad_id is not None else 0
        return [get(c, fallback) for c in codes]

    def decode(self, ids: Iterable[int]) -> List[str]:
        out = []
        for i in ids:
            if 0 <= i < self.vocab_size:
                out.append(self.id2code[i])
            else:
                out.append(self.unk_token or "[UNK]")
        return out

    def save(self, path: str) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        obj = {
            "id2code": self.id2code,
            "special_tokens": [t for t in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"] if t in self.code2id],
            "type2id": self.type2id,
            "pad_id": self.pad_id,
            "mask_id": self.mask_id,
            "cls_id": self.cls_id,
            "unk_id": self.unk_id,
        }
        with open(path, "w") as f:
            json.dump(obj, f, indent=2)

    @classmethod
    def load(cls, path: str) -> "Tokenizer":
        path = Path(path)
        with open(path) as f:
            obj = json.load(f)

        tok = cls.__new__(cls) 

        tok.id2code = obj["id2code"]
        tok.code2id = {tok_: i for i, tok_ in enumerate(tok.id2code)}
        tok.vocab_size = len(tok.id2code)

        tok.special_tokens = obj.get("special_tokens", [])
        tok.type2id = obj.get("type2id", {})

        tok.pad_token  = "[PAD]" if "[PAD]" in tok.code2id else None
        tok.mask_token = "[MASK]" if "[MASK]" in tok.code2id else None
        tok.cls_token  = "[CLS]" if "[CLS]" in tok.code2id else None
        tok.unk_token  = "[UNK]" if "[UNK]" in tok.code2id else None

        tok.pad_id  = obj.get("pad_id", tok.code2id.get("[PAD]", 0))
        tok.mask_id = obj.get("mask_id", tok.code2id.get("[MASK]")) if "[MASK]" in tok.code2id else None
        tok.cls_id  = obj.get("cls_id", tok.code2id.get("[CLS]"))   if "[CLS]" in tok.code2id else None
        tok.unk_id  = obj.get("unk_id", tok.code2id.get("[UNK]"))   if "[UNK]" in tok.code2id else None

        # Rebuild Polars lookup frames
        tok._code2id_df = pl.DataFrame({"code": tok.id2code,
                                        "input_id": list(range(tok.vocab_size))}) \
                              .with_columns(pl.col("code").cast(pl.Categorical))
        tok._type2id_df = pl.DataFrame({"code_type": list(tok.type2id.keys()),
                                        "type_id":   list(tok.type2id.values())}) \
                              .with_columns(pl.col("code_type").cast(pl.Categorical))
        return tok

    @property
    def code2id_df(self) -> pl.DataFrame:
        return self._code2id_df

    @property
    def type2id_df(self) -> pl.DataFrame:
        return self._type2id_df

In [30]:
class SequencesGenerator:
    def __init__(
        self,
        tokenizer_path: str,
        chunk_length: int = 1024,
        overlap: int = 128,
        return_numeric: bool = False,
        return_text: bool = False,
        return_time: bool = False,
        return_ids: bool = False,
        dataset_name: str = "mimic",
    ):

        if tokenizer_path is not None:
            self.tokenizer = Tokenizer.load(tokenizer_path)
            
        self.chunk_length = chunk_length
        self.overlap = overlap
        self.return_numeric = return_numeric
        self.return_text = return_text
        self.return_time = return_time
        self.return_ids = return_ids
        self.dataset_name = dataset_name
        
    def encode_sequence(
        self,
        timeline: pl.DataFrame,
        max_length: Optional[int] = None,
        pad_to_max: bool = False,
        truncation: Literal["head", "tail"] = "tail",
        add_cls: bool = False,
    ) -> Dict[str, Union[List[int], List[float], List[str]]]:
        """
        Vectorized build of:
          input_ids, attention_mask, visit_ids, stage_ids, type_ids
          + optional numeric/text streams (+ masks)
        """
        df = timeline
        
        if "seq_id" in df.columns:

            uniq = df.select(pl.col("seq_id")).unique(maintain_order=True)
            uniq = uniq.with_row_count(name="visit_ids_raw")  # 0..K-1
            df = df.join(uniq, on="seq_id", how="left").with_columns(
                (pl.col("visit_ids_raw") ).alias("visit_id").fill_null(0)
            ).drop("visit_ids_raw")
        else:
            df = df.with_columns(pl.lit(0).alias("visit_id"))

        stage_cols = ["out_id", "er_id", "hadm_id", "icustay_id"]
        present_stages = [c for c in stage_cols if c in df.columns]
        if present_stages:

            expr = pl.lit(0)
            for i, col in enumerate(present_stages, start=1):
                expr = pl.when(expr.eq(0) & pl.col(col).is_not_null()).then(i).otherwise(expr)
            df = df.with_columns(expr.alias("stage_id"))
        else:
            df = df.with_columns(pl.lit(0).alias("stage_id"))

        df = df.join(
            self.tokenizer.type2id_df,
            on=pl.col("code_type").cast(pl.Categorical),
            how="left",
        ).with_columns(pl.col("type_id").fill_null(0))

        df = df.join(
            self.tokenizer.code2id_df,
            on=pl.col("code").cast(pl.Categorical),
            how="left",
        )


        unk_id = self.tokenizer.unk_id if self.tokenizer.unk_id is not None else self.tokenizer.pad_id or 0
        df = df.with_columns(pl.col("input_id").fill_null(unk_id))

        df = df.with_columns(
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("visit_id"))
              .alias("visit_id"),
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("stage_id"))
              .alias("stage_id"),
        )


        if add_cls and (self.tokenizer.cls_token is not None):
            cls_row = {
                "code": self.tokenizer.cls_token,
                "code_type": "[CLS]",
                "visit_id": 0,
                "stage_id": 0,
                "type_id": self.tokenizer.type2id.get("[CLS]", 0),
                "input_id": self.tokenizer.cls_id,
            }

            if self.return_numeric:
                cls_row["numeric_value"] = None
            if self.return_text:
                cls_row["text_value"] = None

            df = pl.concat([pl.DataFrame([cls_row]), df], how="vertical_relaxed")


        input_ids = df.get_column("input_id").cast(pl.Int64).to_list()
        type_ids = df.get_column("type_id").cast(pl.Int64).to_list()
        visit_ids = df.get_column("visit_id").cast(pl.Int64).to_list()
        stage_ids = df.get_column("stage_id").cast(pl.Int64).to_list()
        attention_mask = [1] * len(input_ids)


        value_payload = self._build_value_streams(
            df=df,
            max_length=max_length,
            pad_to_max=pad_to_max,
            truncation=truncation,
        )

        # --- truncate/pad core streams in one go ---
        input_ids      = self._truncate(input_ids,      max_length, truncation)
        type_ids       = self._truncate(type_ids,       max_length, truncation)
        visit_ids      = self._truncate(visit_ids,      max_length, truncation)
        stage_ids      = self._truncate(stage_ids,      max_length, truncation)
        attention_mask = [1] * len(input_ids)

        if pad_to_max and max_length is not None and len(input_ids) < max_length:
            pad_len = max_length - len(input_ids)
            pad_id = self.tokenizer.pad_id if self.tokenizer.pad_id is not None else 0
            input_ids      = input_ids + [pad_id] * pad_len
            type_ids       = type_ids + [0] * pad_len
            visit_ids      = visit_ids + [0] * pad_len
            stage_ids      = stage_ids + [0] * pad_len
            attention_mask = attention_mask + [0] * pad_len

        out = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "visit_ids": visit_ids,
            "stage_ids": stage_ids,
            "type_ids": type_ids,
        }
        out.update(value_payload)
        return out

    def get_overlapped_chunks(
        self,
        timeline: Dict[str, Iterable],
        chunk_length: Optional[int] = None,
        overlap: Optional[int] = None,
        add_cls_per_chunk: bool = True,
    ) -> List[Dict[str, List[Any]]]:

        if chunk_length is None:
            chunk_length = self.chunk_length
        if overlap is None:
            overlap = self.overlap

        if getattr(self, "dataset_name", None) == "ehrshot":
            fields = [
                "tokens",
                "valid_tokens",
                "ages",
                "normalized_ages",
                "timestamps",
            ]

            n = len(timeline["tokens"])
            if n == 0:
                return []

            payload = chunk_length
            step = max(1, payload - overlap)

            num_chunks = 1 if n <= payload else ceil((n - payload) / step) + 1
            starts = [i * step for i in range(num_chunks)]

            chunks = []
            for start in starts:
                end = min(n, start + payload)

                sliced = {
                    k: list(timeline[k][start:end])
                    for k in fields
                    if k in timeline
                }

                chunk_len = len(sliced["tokens"])

                sliced["valid_tokens"] = [True] * chunk_len
                sliced["patient_lengths"] = [chunk_len]
                sliced["label_indices"] = []

                chunks.append(sliced)

            return chunks

        fields = ["input_ids", "attention_mask", "visit_ids", "stage_ids", "type_ids"]
        for extra in ("numeric_values", "numeric_mask", "text_values", "text_mask", "time_diff"):
            if extra in timeline and extra not in fields:
                fields.append(extra)

        n = len(timeline["input_ids"])
        payload = chunk_length - (1 if add_cls_per_chunk else 0)
        step = max(1, payload - overlap)

        num_chunks = 1 if n <= payload else ceil((n - payload) / step) + 1
        starts = [i * step for i in range(num_chunks)]

        cls_defaults = {
            "input_ids": self.tokenizer.cls_id if self.tokenizer.cls_id is not None else (self.tokenizer.pad_id or 0),
            "attention_mask": 1,
            "visit_ids": 0,
            "stage_ids": 0,
            "type_ids": self.tokenizer.type2id.get("[CLS]", 0),
            "numeric_values": 0.0,
            "numeric_mask": 0,
            "text_values": "",
            "text_mask": 0,
            "time_diff": 0.0,
        }

        chunks = []
        for start in starts:
            end = min(n, start + payload)
            sliced = {k: list(timeline[k][start:end]) for k in fields if k in timeline}

            if "attention_mask" in sliced:
                sliced["attention_mask"] = [1] * len(sliced["input_ids"])

            if add_cls_per_chunk:
                for k in list(sliced.keys()):
                    sliced[k] = [cls_defaults[k]] + sliced[k]

            cur_len = len(sliced["input_ids"])
            if cur_len < chunk_length:
                pad_len = chunk_length - cur_len
                for k in list(sliced.keys()):
                    sliced[k] = self._pad_list(sliced[k], pad_len, 0)

            chunks.append(sliced)

        return chunks


    def _build_value_streams(
        self,
        df: pl.DataFrame,
        max_length: Optional[int],
        pad_to_max: bool,
        truncation: Literal["head", "tail"],
    ) -> Dict[str, List[Any]]:
        out: Dict[str, List[Any]] = {}
        # Numeric stream
        if self.return_numeric:
            if "numeric_value" in df.columns:
                vals = df.get_column("numeric_value").to_list()
            else:
                vals = [None] * df.height
            num_mask = [1 if (v is not None) else 0 for v in vals]
            vals = [0.0 if v is None else float(v) for v in vals]

            vals = self._truncate(vals, max_length, truncation)
            num_mask = self._truncate(num_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(vals) < max_length:
                pad_len = max_length - len(vals)
                vals += [0.0] * pad_len
                num_mask += [0] * pad_len

            out["numeric_values"] = vals
            out["numeric_mask"] = num_mask

        # Text stream
        if self.return_text:
            if "text_value" in df.columns:
                txt = df.get_column("text_value").to_list()
            else:
                txt = [None] * df.height
            txt = [("" if (t is None or str(t) == "___") else str(t)) for t in txt]
            txt_mask = [1 if (t != "") else 0 for t in txt]

            txt = self._truncate(txt, max_length, truncation)
            txt_mask = self._truncate(txt_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(txt) < max_length:
                pad_len = max_length - len(txt)
                txt += [""] * pad_len
                txt_mask += [0] * pad_len

            out["text_values"] = txt
            out["text_mask"] = txt_mask

            
        if self.return_time:
            if "time_diff" in df.columns:
                df = df.with_columns(pl.col(['time_diff'])).fill_null(0.0)
                time_diff = df.get_column("time_diff").to_list()
                time_diff = self._scale_time_deltas(time_diff)
                time_stamp = df.get_column("time").to_list()
            else:
                time_diff = [None] * df.height
                time_stamp = [None] * df.height


            if pad_to_max and max_length is not None and len(time_diff) < max_length:
                pad_len = max_length - len(time_diff)
                time_diff += [0] * pad_len
                time_stamp += [0] * pad_len


            out["time_diff"] = time_diff
            out["time_stamp"] = time_stamp
            
        if self.return_ids:
            if "seq_id" in df.columns:
                
                seq_id = df.get_column("seq_id").cast(pl.Int32).to_list()
                out_id = df.get_column("out_id").cast(pl.Int32).to_list()
                er_id =  df.get_column("er_id").cast(pl.Int32).to_list()
                hadm_id = df.get_column("hadm_id").cast(pl.Int32).to_list()
                icustay_id = df.get_column("hadm_id").cast(pl.Int32).to_list()
            else:
                seq_id = [None] * df.height
                out_id = [None] * df.height
                er_id =  [None] * df.height
                hadm_id = [None] * df.height
                icustay_id = [None] * df.height

            if pad_to_max and max_length is not None and len(time_diff) < max_length:
                pad_len = max_length - len(time_diff)
                seq_id += [0] * pad_len
                out_id += [0] * pad_len
                er_id += [0] * pad_len
                hadm_id += [0] * pad_len
                icustay_id += [0] * pad_len

            out["seq_id"] = seq_id
            out["out_id"] = out_id
            out["er_id"] = er_id
            out["hadm_id"] = hadm_id
            out["icustay_id"] = icustay_id

        return out
    
    def get_time_based_chunks(
        self,
        timeline: Dict[str, Iterable],
        window_hours: float,
        anchor_from_first_valid_time: bool = True,
        keep_prefix_tokens: bool = True,
    ) -> List[Dict[str, List[Any]]]:

        if getattr(self, "dataset_name", None) == "ehrshot":
            if "tokens" not in timeline:
                raise ValueError("timeline must contain 'tokens'")
            if "timestamps" not in timeline:
                raise ValueError("timeline must contain 'timestamps' for EHRShot time-based chunking")

            n = len(timeline["tokens"])
            if n == 0:
                return []

            fields = [
                "tokens",
                "valid_tokens",
                "ages",
                "normalized_ages",
                "timestamps",
            ]

            time_stamps = list(timeline["timestamps"])

            first_clinical_idx = None
            for i in range(1, n):
                if time_stamps[i] > time_stamps[i - 1]:
                    if i > 0 and (time_stamps[i] - time_stamps[i - 1]) > 365 * 24 * 3600:
                        first_clinical_idx = i
                        break

            if first_clinical_idx is None:
                # Fallback: no obvious prefix/clinical split.
                first_clinical_idx = 0

            prefix_idx = list(range(first_clinical_idx)) if keep_prefix_tokens else []

            t0 = time_stamps[first_clinical_idx]
            window_size = window_hours * 3600.0

            if window_size <= 0:
                raise ValueError("window_hours must be > 0")

            buckets: Dict[int, List[int]] = {}

            for i in range(first_clinical_idx, n):
                elapsed = time_stamps[i] - t0
                window_id = int(elapsed // window_size) if elapsed >= 0 else 0
                buckets.setdefault(window_id, []).append(i)

            raw_chunks: List[Dict[str, List[Any]]] = []

            for chunk_idx, window_id in enumerate(sorted(buckets.keys())):
                if chunk_idx == 0:
                    idxs = prefix_idx + buckets[window_id]
                else:
                    idxs = buckets[window_id]

                chunk = {
                    k: [timeline[k][j] for j in idxs]
                    for k in fields
                    if k in timeline
                }

                chunk_len = len(chunk["tokens"])
                chunk["valid_tokens"] = [True] * chunk_len
                chunk["patient_lengths"] = [chunk_len]
                chunk["label_indices"] = []

                raw_chunks.append(chunk)

            final_chunks: List[Dict[str, List[Any]]] = []

            for chunk in raw_chunks:
                if len(chunk["tokens"]) <= self.chunk_length:
                    final_chunks.append(chunk)
                else:
                    sub_chunks = self.get_overlapped_chunks(
                        timeline=chunk,
                        chunk_length=self.chunk_length,
                        overlap=0,
                        add_cls_per_chunk=False,
                    )
                    final_chunks.extend(sub_chunks)

            return final_chunks


        if "input_ids" not in timeline:
            raise ValueError("timeline must contain 'input_ids'")
        if "time_stamp" not in timeline:
            raise ValueError("timeline must contain 'time_stamp' for time-based chunking")

        n = len(timeline["input_ids"])
        if n == 0:
            return []

        fields = [k for k, v in timeline.items() if isinstance(v, (list, tuple))]
        time_stamps = list(timeline["time_stamp"])

        def _is_valid_time(x: Any) -> bool:
            return x is not None

        anchor_idx = None
        for i, ts in enumerate(time_stamps):
            if _is_valid_time(ts):
                anchor_idx = i
                break

        if anchor_idx is None:
            return [{k: list(v) for k, v in timeline.items() if isinstance(v, (list, tuple))}]

        t0 = time_stamps[anchor_idx]
        window_size = window_hours * 3600.0

        if window_size <= 0:
            raise ValueError("window_hours must be > 0")

        prefix_idx = list(range(anchor_idx)) if keep_prefix_tokens else []

        buckets: Dict[int, List[int]] = {}
        for i in range(anchor_idx, n):
            ts = time_stamps[i]

            if not _is_valid_time(ts):
                window_id = 0
            else:
                elapsed = (ts - t0).total_seconds()
                window_id = int(elapsed // window_size) if elapsed >= 0 else 0

            buckets.setdefault(window_id, []).append(i)

        chunks: List[Dict[str, List[Any]]] = []
        for chunk_idx, window_id in enumerate(sorted(buckets.keys())):
            idxs = (prefix_idx + buckets[window_id]) if chunk_idx == 0 else buckets[window_id]

            chunk = {}
            for k in fields:
                chunk[k] = [timeline[k][j] for j in idxs]

            chunks.append(chunk)

        final_chunks = []

        for chunk in chunks:
            sub_chunks = self.get_overlapped_chunks(
                timeline=chunk,
                chunk_length=self.chunk_length,
                overlap=0,
                add_cls_per_chunk=True,
            )
            final_chunks.extend(sub_chunks)

        return final_chunks
    
    def get_visit_level_chunks(
        self,
        timeline: Dict[str, Iterable],
        keep_prefix_tokens: bool = True,
    ) -> List[Dict[str, List[Any]]]:

        if "input_ids" not in timeline:
            raise ValueError("timeline must contain 'input_ids'")
        if "seq_id" not in timeline:
            raise ValueError("timeline must contain 'seq_id' for visit-level chunking")

        n = len(timeline["input_ids"])
        if n == 0:
            return []

        fields = [k for k, v in timeline.items() if isinstance(v, (list, tuple))]
        seq_ids = list(timeline["seq_id"])

        def _is_valid_visit(x: Any) -> bool:
            return x is not None and x != 0

        # find first actual visit event
        first_visit_idx = None
        for i, sid in enumerate(seq_ids):
            if _is_valid_visit(sid):
                first_visit_idx = i
                break

        # if no valid seq_id exists, return overlapped chunks on full sequence
        if first_visit_idx is None:
            return self.get_overlapped_chunks(
                timeline={k: list(v) for k, v in timeline.items() if isinstance(v, (list, tuple))},
                chunk_length=self.chunk_length,
                overlap=0,
                add_cls_per_chunk=True,
            )

        prefix_idx = list(range(first_visit_idx)) if keep_prefix_tokens else []

        buckets: Dict[Any, List[int]] = {}
        visit_order: List[Any] = []

        for i in range(first_visit_idx, n):
            sid = seq_ids[i]
            if not _is_valid_visit(sid):
                continue

            if sid not in buckets:
                buckets[sid] = []
                visit_order.append(sid)
            buckets[sid].append(i)

        raw_chunks: List[Dict[str, List[Any]]] = []
        for chunk_idx, sid in enumerate(visit_order):
            idxs = (prefix_idx + buckets[sid]) if chunk_idx == 0 else buckets[sid]

            chunk = {}
            for k in fields:
                chunk[k] = [timeline[k][j] for j in idxs]

            raw_chunks.append(chunk)

        final_chunks: List[Dict[str, List[Any]]] = []
        for chunk in raw_chunks:
            sub_chunks = self.get_overlapped_chunks(
                timeline=chunk,
                chunk_length=self.chunk_length,
                overlap=0,
                add_cls_per_chunk=True,
            )
            final_chunks.extend(sub_chunks)

        return final_chunks

    
    def get_care_stage_level_chunks(
        self,
        timeline: Dict[str, Iterable],
        keep_prefix_tokens: bool = True,
    ) -> List[Dict[str, List[Any]]]:

        if "input_ids" not in timeline:
            raise ValueError("timeline must contain 'input_ids'")
        if "seq_id" not in timeline:
            raise ValueError("timeline must contain 'seq_id' for care-stage chunking")

        n = len(timeline["input_ids"])
        if n == 0:
            return []

        fields = [k for k, v in timeline.items() if isinstance(v, (list, tuple))]

        seq_ids = list(timeline["seq_id"]) if "seq_id" in timeline else [None] * n
        er_ids = list(timeline["er_id"]) if "er_id" in timeline else [None] * n
        out_ids = list(timeline["out_id"]) if "out_id" in timeline else [None] * n
        hadm_ids = list(timeline["hadm_id"]) if "hadm_id" in timeline else [None] * n
        icu_ids = list(timeline["icustay_id"]) if "icustay_id" in timeline else [None] * n

        def _valid(x: Any) -> bool:
            return x is not None and x != 0

        # make hadm mutually exclusive with icu at the row level
        hadm_ids = [
            None if _valid(icu) else hadm
            for hadm, icu in zip(hadm_ids, icu_ids)
        ]

        # per-row stage priority
        def _stage_key(i: int):
            if _valid(icu_ids[i]):
                return ("icu", icu_ids[i])
            elif _valid(hadm_ids[i]):
                return ("hadm", hadm_ids[i])
            elif _valid(er_ids[i]):
                return ("er", er_ids[i])
            elif _valid(out_ids[i]):
                return ("out", out_ids[i])
            return None

        # first row that belongs to a visit
        first_visit_idx = None
        for i in range(n):
            if _valid(seq_ids[i]):
                first_visit_idx = i
                break

        if first_visit_idx is None:
            return self.get_overlapped_chunks(
                timeline={k: list(v) for k, v in timeline.items() if isinstance(v, (list, tuple))},
                chunk_length=self.chunk_length,
                overlap=0,
                add_cls_per_chunk=True,
            )

        prefix_idx = list(range(first_visit_idx)) if keep_prefix_tokens else []

        # group by (seq_id, stage_type, stage_id), preserving first-seen order
        buckets: Dict[Any, List[int]] = {}
        chunk_order: List[Any] = []

        for i in range(first_visit_idx, n):
            if not _valid(seq_ids[i]):
                continue

            stage = _stage_key(i)
            if stage is None:
                continue

            key = (seq_ids[i], stage[0], stage[1])

            if key not in buckets:
                buckets[key] = []
                chunk_order.append(key)
            buckets[key].append(i)

        final_chunks: List[Dict[str, List[Any]]] = []

        for chunk_idx, key in enumerate(chunk_order):
            idxs = (prefix_idx + buckets[key]) if chunk_idx == 0 else buckets[key]

            chunk = {}
            for k in fields:
                chunk[k] = [timeline[k][j] for j in idxs]

            sub_chunks = self.get_overlapped_chunks(
                timeline=chunk,
                chunk_length=self.chunk_length,
                overlap=0,
                add_cls_per_chunk=True,
            )

#             for sub_chunk in sub_chunks:
#                 sub_chunk["source_seq_id"] = key[0]
#                 sub_chunk["source_stage_type"] = key[1]
#                 sub_chunk["source_stage_id"] = key[2]

            final_chunks.extend(sub_chunks)

        return final_chunks
    
    def _scale_time_deltas(self, deltas_list):
        deltas = np.asarray(deltas_list, dtype=float)
        compressed = np.log1p(deltas)              
        scaled = compressed / np.log(5328.93125)         
        return scaled.tolist()

    @staticmethod
    def _truncate(seq: List[Any], max_length: Optional[int], truncation: str) -> List[Any]:
        if max_length is None or len(seq) <= max_length:
            return seq
        return seq[-max_length:] if truncation == "head" else seq[:max_length]

    @staticmethod
    def _pad_list(lst: List[Any], pad_len: int, pad_value: Any) -> List[Any]:
        if pad_len <= 0:
            return lst
        return lst + [pad_value] * pad_len

In [31]:
seq_gen = SequencesGenerator(tokenizer_path=None,
                             chunk_length=256,
                             overlap=32,
                             dataset_name='ehrshot'
                             )

In [32]:
arrow_ds = load_from_disk('ehrshot_clmbr_tokenized_arrow')

In [33]:
# clmbr_idx.filter(pl.col('arrow_row_idx') == 0)

In [34]:
# chunks = seq_gen.get_time_based_chunks(
#     timeline=arrow_ds[1]["transformer"],
#     window_hours=24.0*30,
# )

# [(len(c["tokens"]), c["patient_lengths"]) for c in chunks]

In [35]:
# import datetime

# def verify_time_chunks(chunks, window_hours, allow_prefix_first_chunk=True):
#     window_sec = window_hours * 3600

#     for i, ch in enumerate(chunks):
#         ts = ch["timestamps"]

#         if len(ts) == 0:
#             print(f"chunk {i}: empty")
#             continue

#         # First chunk may contain birth/demographic prefix.
#         if i == 0 and allow_prefix_first_chunk:
#             # remove prefix by keeping only timestamps close to clinical range
#             # prefix usually has a very large gap before clinical history
#             diffs = [ts[j] - ts[j - 1] for j in range(1, len(ts))]
#             split = 0
#             for j, d in enumerate(diffs, start=1):
#                 if d > 365 * 24 * 3600:
#                     split = j
#                     break
#             clinical_ts = ts[split:]
#         else:
#             clinical_ts = ts

#         if len(clinical_ts) == 0:
#             print(f"chunk {i}: prefix only")
#             continue

#         span = max(clinical_ts) - min(clinical_ts)

#         ok = span <= window_sec

#         print(
#             f"chunk={i:04d} | "
#             f"n={len(ts):4d} | "
#             f"clinical_n={len(clinical_ts):4d} | "
#             f"span_hr={span / 3600:.2f} | "
#             f"ok={ok}"
#         )

#         if not ok:
#             print("  min:", datetime.datetime.fromtimestamp(min(clinical_ts)))
#             print("  max:", datetime.datetime.fromtimestamp(max(clinical_ts)))
#             raise AssertionError(f"Chunk {i} exceeds {window_hours} hours")

#     print("All chunks passed.")

In [36]:
# clmbr_idx.filter(pl.col('n_clmbr_tokens')>25000)

In [37]:
# import numpy as np
# import polars as pl
# from tqdm import tqdm

# def analyze_time_gaps(arrow_ds, max_patients=None):
#     gaps_sec = []
#     patient_spans_days = []
#     patient_lengths = []

#     n = len(arrow_ds) if max_patients is None else min(max_patients, len(arrow_ds))

#     for i in tqdm(range(n)):
#         tr = arrow_ds[i]["transformer"]

#         ts = np.asarray(tr["timestamps"])

#         if len(ts) < 2:
#             continue

#         diffs = np.diff(ts)

#         gaps_sec.extend(diffs.tolist())

#         patient_spans_days.append((ts[-1] - ts[0]) / 86400)
#         patient_lengths.append(len(ts))

#     gaps_sec = np.asarray(gaps_sec)

#     df = pl.DataFrame({
#         "statistic": [
#             "count",
#             "mean_gap_min",
#             "median_gap_min",
#             "p75_gap_min",
#             "p90_gap_min",
#             "p95_gap_min",
#             "p99_gap_min",
#             "max_gap_days",
#         ],
#         "value": [
#             float(len(gaps_sec)),
#             float(gaps_sec.mean() / 60),
#             float(np.percentile(gaps_sec, 50) / 60),
#             float(np.percentile(gaps_sec, 75) / 60),
#             float(np.percentile(gaps_sec, 90) / 60),
#             float(np.percentile(gaps_sec, 95) / 60),
#             float(np.percentile(gaps_sec, 99) / 60),
#             float(gaps_sec.max() / 86400),
#         ]
#     })

#     patient_df = pl.DataFrame({
#         "patient_length": patient_lengths,
#         "timeline_span_days": patient_spans_days,
#     })

#     return df, patient_df

In [38]:
# gap_stats, patient_stats = analyze_time_gaps(
#     arrow_ds,
#     max_patients=None,
# )

# print(gap_stats)

# print(patient_stats.select("patient_length").describe())
# print(patient_stats.select("timeline_span_days").describe())

In [39]:
# def bucket_size_distribution(
#     arrow_ds,
#     window_hours=6,
#     max_patients=None,
# ):
#     bucket_sizes = []

#     n = len(arrow_ds) if max_patients is None else min(max_patients, len(arrow_ds))

#     window_sec = window_hours * 3600

#     for i in tqdm(range(n)):
#         ts = np.asarray(arrow_ds[i]["transformer"]["timestamps"])

#         if len(ts) == 0:
#             continue

#         # find first large jump = start of clinical history
#         start_idx = 0
#         for j in range(1, len(ts)):
#             if ts[j] - ts[j - 1] > 365 * 24 * 3600:
#                 start_idx = j
#                 break

#         t0 = ts[start_idx]

#         bucket_counts = {}

#         for t in ts[start_idx:]:
#             b = int((t - t0) // window_sec)
#             bucket_counts[b] = bucket_counts.get(b, 0) + 1

#         bucket_sizes.extend(bucket_counts.values())

#     bucket_sizes = np.asarray(bucket_sizes)

#     return pl.DataFrame({
#         "statistic": [
#             "count",
#             "mean",
#             "median",
#             "p75",
#             "p90",
#             "p95",
#             "p99",
#             "max",
#         ],
#         "value": [
#             float(len(bucket_sizes)),
#             float(bucket_sizes.mean()),
#             float(np.percentile(bucket_sizes, 50)),
#             float(np.percentile(bucket_sizes, 75)),
#             float(np.percentile(bucket_sizes, 90)),
#             float(np.percentile(bucket_sizes, 95)),
#             float(np.percentile(bucket_sizes, 99)),
#             float(bucket_sizes.max()),
#         ]
#     })

In [40]:
# for hrs in [6, 12, 24]:
#     print(f"\n===== {hrs} hour buckets =====")
#     print(bucket_size_distribution(
#         arrow_ds,
#         window_hours=hrs,
#         max_patients=10000
#     ))

In [41]:
# import numpy as np
# from tqdm import tqdm

# def fraction_buckets_exceeding_limit(
#     arrow_ds,
#     window_hours=24,
#     chunk_length=256,
# ):
#     window_sec = window_hours * 3600

#     total_buckets = 0
#     oversized_buckets = 0

#     for i in tqdm(range(len(arrow_ds))):

#         ts = np.asarray(arrow_ds[i]["transformer"]["timestamps"])

#         if len(ts) == 0:
#             continue

#         # find first clinical event
#         start_idx = 0
#         for j in range(1, len(ts)):
#             if ts[j] - ts[j - 1] > 365 * 24 * 3600:
#                 start_idx = j
#                 break

#         t0 = ts[start_idx]

#         bucket_counts = {}

#         for t in ts[start_idx:]:
#             bucket_id = int((t - t0) // window_sec)
#             bucket_counts[bucket_id] = bucket_counts.get(bucket_id, 0) + 1

#         counts = np.array(list(bucket_counts.values()))

#         total_buckets += len(counts)
#         oversized_buckets += np.sum(counts > chunk_length)

#     print(f"window_hours = {window_hours}")
#     print(f"chunk_length = {chunk_length}")
#     print(f"total buckets = {total_buckets:,}")
#     print(f"oversized buckets = {oversized_buckets:,}")
#     print(f"fraction oversized = {oversized_buckets / total_buckets:.6%}")

In [42]:
# for hrs in [6, 12, 24]:
#     fraction_buckets_exceeding_limit(
#         arrow_ds,
#         window_hours=hrs,
#         chunk_length=256,
#     )
#     print()

In [43]:
chunks = seq_gen.get_overlapped_chunks(timeline=arrow_ds[0]['transformer'])

In [44]:
len(chunks)

3

In [45]:
# import torch
# import numpy as np

# def pack_clmbr_chunks(chunks, patient_id=0):
#     lengths = [len(c["tokens"]) for c in chunks]

#     return {
#         "num_patients": len(chunks),
#         "num_indices": 0,
#         "patient_ids": torch.tensor(
#             [patient_id for _ in range(sum(lengths))],
#             dtype=torch.long,
#         ),
#         "offsets": torch.zeros(len(chunks), dtype=torch.int32),
#         "transformer": {
#             "tokens": torch.tensor(
#                 sum([c["tokens"] for c in chunks], []),
#                 dtype=torch.long,
#             ),
#             "valid_tokens": torch.tensor(
#                 sum([c["valid_tokens"] for c in chunks], []),
#                 dtype=torch.bool,
#             ),
#             "ages": torch.tensor(
#                 sum([c["ages"] for c in chunks], []),
#                 dtype=torch.float32,
#             ),
#             "normalized_ages": torch.tensor(
#                 sum([c["normalized_ages"] for c in chunks], []),
#                 dtype=torch.float16,
#             ),
#             "timestamps": torch.tensor(
#                 sum([c["timestamps"] for c in chunks], []),
#                 dtype=torch.long,
#             ),
#             "patient_lengths": torch.tensor(lengths, dtype=torch.int32),
#             "label_indices": torch.tensor([], dtype=torch.int32),
#         },
#     }

# raw_batch = pack_clmbr_chunks(chunks)

# batch = batch_processor.collate([raw_batch])

# with torch.no_grad():
#     batch_inner = femr.models.transformer.remove_first_dimension(batch["batch"])
#     reprs = model.transformer(batch_inner["transformer"])

# lengths = torch.as_tensor(
#     raw_batch["transformer"]["patient_lengths"],
#     dtype=torch.long,
# )
# end_positions = torch.cumsum(lengths, dim=0) - 1
# chunk_embs = reprs[end_positions]

# print("chunk lengths:", lengths.tolist())
# print("total tokens:", raw_batch["transformer"]["tokens"].shape)
# print("reprs shape:", reprs.shape)
# print("end positions:", end_positions.tolist())
# print("chunk_embs shape:", chunk_embs.shape)

In [46]:
# for i, chunk in enumerate(chunks):
#     single_raw = pack_clmbr_chunks([chunk])
#     single_batch = batch_processor.collate([single_raw])

#     with torch.no_grad():
#         single_inner = femr.models.transformer.remove_first_dimension(single_batch["batch"])
#         single_reprs = model.transformer(single_inner["transformer"])

#     single_emb = single_reprs[-1]
#     packed_emb = chunk_embs[i]

#     print(
#         i,
#         torch.allclose(single_emb, packed_emb, atol=1e-5),
#         torch.max(torch.abs(single_emb - packed_emb)).item()
#     )

In [47]:
import torch
import torch.nn as nn
from typing import Any, Dict, List, Mapping, Optional


class CLMBREmbedder(nn.Module):
    def __init__(
        self,
        model: nn.Module,
        batch_processor=None,
        normalize: bool = False,
        device: Optional[str] = None,
    ):
        super().__init__()

        self.model = model
        self.batch_processor = batch_processor
        self.normalize = normalize
        self.device_ = device or ("cuda" if torch.cuda.is_available() else "cpu")

        self.model.eval().to(self.device_)

    @torch.no_grad()
    def encode(self, raw_batch: Mapping[str, Any]) -> torch.Tensor:

        if self.batch_processor is not None:
            batch = self.batch_processor.collate([raw_batch])["batch"]
            batch = femr.models.transformer.remove_first_dimension(batch)
        else:
            batch = raw_batch

        transformer_batch = self._move_transformer_to_device(batch["transformer"])

        reprs = self.model.transformer(transformer_batch)

        patient_lengths = torch.as_tensor(
            transformer_batch["patient_lengths"],
            dtype=torch.long,
            device=reprs.device,
        )

        end_positions = torch.cumsum(patient_lengths, dim=0) - 1
        embs = reprs[end_positions]

        if self.normalize:
            embs = nn.functional.normalize(embs, p=2, dim=-1)

        return embs

    def _move_transformer_to_device(
        self,
        transformer_batch: Mapping[str, Any],
    ) -> Dict[str, Any]:
        out = {}

        for k, v in transformer_batch.items():
            if torch.is_tensor(v):
                out[k] = v.to(self.device_)
            else:
                out[k] = torch.as_tensor(v).to(self.device_)

        return out


def pack_clmbr_chunks(
    chunks: List[Dict[str, List[Any]]],
    patient_id: int = 0,
) -> Dict[str, Any]:


    if len(chunks) == 0:
        raise ValueError("Cannot pack empty chunk list.")

    lengths = [len(c["tokens"]) for c in chunks]

    if any(length == 0 for length in lengths):
        raise ValueError("Empty CLMBR chunk found.")

    total_len = sum(lengths)

    def flatten(key):
        return sum([list(c[key]) for c in chunks], [])

    return {
        "num_patients": len(chunks),
        "num_indices": 0,
        "patient_ids": torch.tensor(
            [patient_id] * total_len,
            dtype=torch.long,
        ),
        "offsets": torch.zeros(len(chunks), dtype=torch.int32),
        "transformer": {
            "tokens": torch.tensor(flatten("tokens"), dtype=torch.long),
            "valid_tokens": torch.tensor(flatten("valid_tokens"), dtype=torch.bool),
            "ages": torch.tensor(flatten("ages"), dtype=torch.float32),
            "normalized_ages": torch.tensor(flatten("normalized_ages"), dtype=torch.float16),
            "timestamps": torch.tensor(flatten("timestamps"), dtype=torch.long),
            "patient_lengths": torch.tensor(lengths, dtype=torch.int32),
            "label_indices": torch.tensor([], dtype=torch.int32),
        },
    }

In [48]:
# embedder = CLMBREmbedder(
#     model=model,
#     batch_processor=batch_processor,
#     normalize=False,
# )

# raw_batch = pack_clmbr_chunks(chunks, patient_id=115967095)
# chunk_embs = embedder.encode(raw_batch)

In [49]:
# raw_batch

In [50]:
import faiss
import torch
import numpy as np
from collections import defaultdict

In [51]:
def to_list(v):
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, list):
        return v
    return v

In [52]:
def build_faiss_index(ch_embs, metric: str = "l2"):

    # Accept either:
    #  - list[Tensor] each (1,D) or (D,)
    #  - Tensor of shape (N,D) or (D,) or (1,D)
    if torch.is_tensor(ch_embs):
        v = ch_embs.detach().cpu()
        if v.ndim == 1:
            xb = v.unsqueeze(0).numpy()
        elif v.ndim == 2:
            xb = v.numpy()
        else:
            raise ValueError(f"ch_embs tensor must be (D,) or (N,D). Got {tuple(v.shape)}")
    else:
        xb_list = []
        for t in ch_embs:
            if not torch.is_tensor(t):
                raise TypeError(f"Expected torch.Tensor embeddings, got {type(t)}")
            v = t.detach().cpu()
            if v.ndim == 2 and v.shape[0] == 1:
                v = v[0]
            elif v.ndim != 1:
                raise ValueError(f"Each embedding must be (D,) or (1,D). Got {tuple(v.shape)}")
            xb_list.append(v.numpy())
        xb = np.stack(xb_list, axis=0)

    xb = xb.astype(np.float32, copy=False)
    xb = np.ascontiguousarray(xb)  # <-- FIX

    dim = xb.shape[1]
    metric = metric.lower()

    if metric in ("cosine", "ip", "inner_product", "dot"):
        faiss.normalize_L2(xb)      # now safe
        index = faiss.IndexFlatIP(dim)
    elif metric in ("l2", "euclidean"):
        index = faiss.IndexFlatL2(dim)
    else:
        raise ValueError(f"Unknown metric={metric}. Use 'l2' or 'cosine'.")

    index.add(xb)
    return index, xb

In [53]:
def build_clmbr_indices(
    data_idx_path: str,
    arrow_dataset_path: str,
    task: str,
    save_path: str,
    model,
    batch_processor,
    query_length: int = 512,
    history_chunk_length: int = 256,
    history_overlap: int = 0,
    chunking_strategy: str = "overlap",
    window_hours: float = 6.0,
):

    assert chunking_strategy in ["overlap", "time"]

#     os.makedirs(save_path, exist_ok=True)

    arrow_ds = load_from_disk(arrow_dataset_path)
    data_idx = pl.read_parquet(data_idx_path)
    data_idx = data_idx.filter(pl.col('task') == task)

    seq_gen = SequencesGenerator(
        tokenizer_path=None,
        chunk_length=history_chunk_length,
        overlap=history_overlap,
        dataset_name="ehrshot")

    embedder = CLMBREmbedder(model=model, batch_processor=batch_processor, normalize=False)

    for row_i in tqdm(range(len(data_idx))):
        row = data_idx[row_i]
        subject_id = int(row["subject_id"][0])
        example_id = int(row["example_id"][0])
        arrow_row_idx = int(row["arrow_row_idx"][0])
        prediction_idx = int(row["prediction_idx"][0])
        history_min_idx = int(row["history_min_idx"][0])
        sample = arrow_ds[arrow_row_idx]
        tr = sample["transformer"]

        timeline = {
            "tokens": to_list(tr["tokens"]),
            "valid_tokens": to_list(tr["valid_tokens"]),
            "ages": to_list(tr["ages"]),
            "normalized_ages": to_list(tr["normalized_ages"]),
            "timestamps": to_list(tr["timestamps"]),
            "patient_lengths": to_list(tr["patient_lengths"]),
            "label_indices": to_list(tr["label_indices"])}

        query_start = max(history_min_idx, prediction_idx - query_length)
        query_end = prediction_idx

        query = {k: v[query_start:query_end] for k, v in timeline.items() if isinstance(v, list)}

        query["patient_lengths"] = [len(query["tokens"])]
        query["label_indices"] = []

        query_batch = pack_clmbr_chunks([query])
        q_emb = embedder.encode(query_batch)

        history_start = history_min_idx
        history_end = query_start

        history = {k: v[history_start:history_end] for k, v in timeline.items() if isinstance(v, list)}

        history_chunks = []

        if len(history["tokens"]) > 0:
            if chunking_strategy == "overlap":
                history_chunks = seq_gen.get_overlapped_chunks(timeline=history, add_cls_per_chunk=False)

            elif chunking_strategy == "time":
                history_chunks = seq_gen.get_time_based_chunks(timeline=history, window_hours=window_hours)

        if len(history_chunks) > 0:
            history_batch = pack_clmbr_chunks(history_chunks)
            h_emb = embedder.encode(history_batch)
            all_embs = torch.cat([h_emb, q_emb], dim=0)

        else:
            all_embs = q_emb

        index, _ = build_faiss_index(ch_embs=all_embs,metric="cosine")
        faiss_name = f"{subject_id}_{example_id}.faiss"
        faiss.write_index(index,os.path.join(save_path, faiss_name))

In [54]:
# build_clmbr_indices(
#     data_idx_path='./clmbr_idx.parquet',
#     arrow_dataset_path="./ehrshot_clmbr_tokenized_arrow",
#     save_path="./test/",
#     task='new_hypertension',
#     model=model,
#     batch_processor=batch_processor,
#     query_length=512,
#     history_chunk_length=256,
#     history_overlap=32,
#     chunking_strategy="overlap",
#     window_hours=24.0,
# )

In [55]:
import os
import faiss
import numpy as np
import polars as pl
from datasets import load_from_disk
from torch.utils.data import Dataset


class CLMBRRetrievalDataset(Dataset):
    def __init__(
        self,
        dataset_path: str,
        data_idx_path: str,
        vectordb_path: str,
        task: str,
        split: str = "train",
        top_k: int = 8,
        query_length: int = 512,
        history_chunk_length: int = 256,
        history_overlap: int = 0,
        chunking_strategy: str = "overlap",
        window_hours: float = 24.0,
    ) -> None:

        assert chunking_strategy in ["overlap", "time"]

        self.vectordb_path = vectordb_path
        self.task = task
        self.top_k = top_k
        self.query_length = query_length
        self.chunking_strategy = chunking_strategy
        self.window_hours = window_hours

        self.arrow_ds = load_from_disk(dataset_path)
        
        splits = {"train":"train",
                  "val":"tuning",
                  "test":"held_out"}
        split = splits[split]
        
        self.data_idx = (
            pl.read_parquet(data_idx_path)
            .filter((pl.col("task") == task) & (pl.col("split") == split))
        )

        self.history_gen = SequencesGenerator(
            tokenizer_path=None,
            chunk_length=history_chunk_length,
            overlap=history_overlap,
            dataset_name="ehrshot",
        )

    def __len__(self) -> int:
        return self.data_idx.height

    def _to_list(self, v):
        if isinstance(v, np.ndarray):
            return v.tolist()
        if hasattr(v, "tolist"):
            return v.tolist()
        if isinstance(v, list):
            return v
        return v

    def _get_label(self, row):
        for col in ["boolean_value", "integer_value", "float_value", "categorical_value"]:
            value = row[col][0]
            if value is not None:
                return value
        raise ValueError("No valid label value found.")

    def _slice_timeline(self, timeline, start: int, end: int):
        out = {
            k: v[start:end]
            for k, v in timeline.items()
            if isinstance(v, list)
        }

        if "tokens" in out:
            n = len(out["tokens"])
            out["valid_tokens"] = [True] * n
            out["patient_lengths"] = [n]
            out["label_indices"] = []

        return out

    def __getitem__(self, idx: int):
        row = self.data_idx[idx]

        subject_id = int(row["subject_id"][0])
        example_id = int(row["example_id"][0])
        arrow_row_idx = int(row["arrow_row_idx"][0])

        prediction_idx = int(row["prediction_idx"][0])
        history_min_idx = int(row["history_min_idx"][0])

        label = self._get_label(row)

        sample = self.arrow_ds[arrow_row_idx]
        tr = sample["transformer"]

        timeline = {
            "tokens": self._to_list(tr["tokens"]),
            "valid_tokens": self._to_list(tr["valid_tokens"]),
            "ages": self._to_list(tr["ages"]),
            "normalized_ages": self._to_list(tr["normalized_ages"]),
            "timestamps": self._to_list(tr["timestamps"]),
        }

        query_start = max(history_min_idx, prediction_idx - self.query_length)
        query_end = prediction_idx

        history_start = history_min_idx
        history_end = query_start

        query = self._slice_timeline(timeline, query_start, query_end)

        history_timeline = self._slice_timeline(timeline, history_start, history_end)

        if len(history_timeline["tokens"]) == 0:
            history_chunks = []
        elif self.chunking_strategy == "overlap":
            history_chunks = self.history_gen.get_overlapped_chunks(
                timeline=history_timeline,
                add_cls_per_chunk=False,
            )
        elif self.chunking_strategy == "time":
            history_chunks = self.history_gen.get_time_based_chunks(
                timeline=history_timeline,
                window_hours=self.window_hours,
            )

        faiss_path = os.path.join(
            self.vectordb_path,
            f"{subject_id}_{example_id}.faiss",
        )

        history_index = faiss.read_index(faiss_path)

        qid = history_index.ntotal - 1
        query_embed = history_index.reconstruct(qid)

        _, ids = self._query_faiss(
            index=history_index,
            q_emb=query_embed,
            top_k=self.top_k,
            metric="cosine",
        )

        ids = ids[0].tolist()

        ids = [
            i for i in ids
            if (i != -1) and (i != qid) and (0 <= i < len(history_chunks))
        ]

        retrieved_history = [history_chunks[i] for i in ids]

        return {
            "query": query,
            "history": retrieved_history,
            "label": label,
            "subject_id": subject_id,
            "example_id": example_id,
        }

    def _query_faiss(self, index, q_emb, top_k: int = 10, metric: str = "cosine"):
        xq = q_emb

        if xq.ndim == 1:
            xq = xq.reshape(1, -1)
        elif xq.ndim != 2:
            raise ValueError(f"q_emb must be (D,) or (B,D). Got {xq.shape}")

        xq = xq.astype(np.float32, copy=False)
        xq = np.ascontiguousarray(xq)

        metric = metric.lower()

        if metric in ("cosine", "ip", "inner_product", "dot"):
            scores, ids = index.search(xq, top_k)
            return scores, ids

        if metric in ("l2", "euclidean"):
            dists, ids = index.search(xq, top_k)
            return dists, ids

        raise ValueError(f"Unknown metric={metric}. Use 'l2' or 'cosine'.")

In [56]:
tds = CLMBRRetrievalDataset(
    dataset_path='./ehrshot_clmbr_tokenized_arrow/',
    data_idx_path='./clmbr_idx.parquet',
    vectordb_path='./faiss_clmbr/512/overlap/256/new_hyperlipidemia',
    task="new_hyperlipidemia",
    split="train",
    top_k=24,
    query_length=512,
    history_chunk_length=256,
    history_overlap=32,
    chunking_strategy="overlap",
)


vds = CLMBRRetrievalDataset(
    dataset_path='./ehrshot_clmbr_tokenized_arrow/',
    data_idx_path='./clmbr_idx.parquet',
    vectordb_path='./faiss_clmbr/512/overlap/256/new_hyperlipidemia',
    task="new_hyperlipidemia",
    split="val",
    top_k=24,
    query_length=512,
    history_chunk_length=256,
    history_overlap=32,
    chunking_strategy="overlap",
)


teds = CLMBRRetrievalDataset(
    dataset_path='./ehrshot_clmbr_tokenized_arrow/',
    data_idx_path='./clmbr_idx.parquet',
    vectordb_path='./faiss_clmbr/512/overlap/256/new_hyperlipidemia',
    task="new_hyperlipidemia",
    split="test",
    top_k=24,
    query_length=512,
    history_chunk_length=256,
    history_overlap=32,
    chunking_strategy="overlap",
)

In [57]:
import torch
from typing import Any, Dict, List


class CLMBRRetrievalCollator:
    def __init__(self, top_k: int):
        self.top_k = top_k

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        query_chunks = [b["query"] for b in batch]

        query_batch = pack_clmbr_chunks(
            query_chunks,
            patient_id=0,
        )

        retrieved_lists = [list(b["history"]) for b in batch]

        valid_masks = []
        padded_history_chunks = []

        for i, retrieved in enumerate(retrieved_lists):
            n_real = len(retrieved)

            if n_real > self.top_k:
                retrieved = retrieved[: self.top_k]
                n_real = self.top_k

            if n_real == 0:
                template = query_chunks[i]
            else:
                template = retrieved[0]

            dummy = self._make_dummy_chunk(template)

            if n_real < self.top_k:
                retrieved = retrieved + [dummy] * (self.top_k - n_real)

            valid_masks.append([1] * n_real + [0] * (self.top_k - n_real))
            padded_history_chunks.extend(retrieved)

        history_batch = pack_clmbr_chunks(
            padded_history_chunks,
            patient_id=0,
        )

        labels = torch.tensor(
            [self._label_to_float(b["label"]) for b in batch],
            dtype=torch.float32,
        )

        history_valid_mask = torch.tensor(
            valid_masks,
            dtype=torch.long,
        )

        return {
            "query": query_batch,
            "history": history_batch,
            "history_valid_mask": history_valid_mask,
            "label": labels,
        }

    def _make_dummy_chunk(self, template: Dict[str, List[Any]]) -> Dict[str, List[Any]]:
        """
        Create a minimal one-token dummy CLMBR chunk.

        It is not meant to carry information.
        It only preserves B*K packed sequence structure.
        Downstream modules must ignore it using history_valid_mask.
        """

        dummy = {
            "tokens": [0],
            "valid_tokens": [True],
            "ages": [0.0],
            "normalized_ages": [0.0],
            "timestamps": [0],
            "patient_lengths": [1],
            "label_indices": [],
        }

        return dummy

    @staticmethod
    def _label_to_float(label):
        if isinstance(label, bool):
            return float(label)

        if label is None:
            raise ValueError("Label is None.")

        return float(label)

In [59]:
from torch.utils.data import DataLoader
collator = CLMBRRetrievalCollator(top_k=8)

tloader = DataLoader(
    tds,
    batch_size=4,
    shuffle=True,
    collate_fn=collator,
)


vloader = DataLoader(
    vds,
    batch_size=4,
    shuffle=True,
    collate_fn=collator,
)

teloader = DataLoader(
    teds,
    batch_size=4,
    shuffle=True,
    collate_fn=collator,
)
batch = next(iter(tloader))

print(batch["query"]["transformer"]["patient_lengths"])
print(batch["history"]["transformer"]["patient_lengths"].shape)
print(batch["history_valid_mask"].shape)
print(batch["label"].shape)

tensor([512, 512, 512, 477], dtype=torch.int32)
torch.Size([32])
torch.Size([4, 8])
torch.Size([4])


In [80]:
import inspect
import femr.models.transformer

print(inspect.getsource(femr.models.transformer.CLMBRTaskHead))

class CLMBRTaskHead(nn.Module):
    def __init__(self, hidden_size: int, clmbr_vocab_size: int):
        super().__init__()

        self.final_layer = nn.Linear(hidden_size, clmbr_vocab_size)

    def forward(self, features: torch.Tensor, batch: Mapping[str, torch.Tensor], return_logits=False):
        logits = self.final_layer(features)
        labels = batch["labels"]
        loss = F.cross_entropy(logits, labels)

        if not return_logits:
            logits = None

        return loss, {"logits": logits}



(features: 'torch.Tensor', batch: 'Mapping[str, torch.Tensor]', return_logits=False)


In [61]:
import os
import torch


import lightning as lt
import torch.nn as nn
import torch.nn.functional as F
from typing import Any, Dict, Optional
from torchmetrics.classification import Accuracy, BinaryAUROC, BinaryAveragePrecision


In [62]:
import os
import yaml
import torch
import wandb
import polars as pl
from tqdm import tqdm
import torch.nn as nn
import torch.distributed as dist

from typing import Callable
from transformers import BertConfig
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision
from transformers import CONFIG_MAPPING, MODEL_FOR_MASKED_LM_MAPPING, MODEL_MAPPING, MODEL_FOR_CAUSAL_LM_MAPPING

In [63]:
BERT_VARIANTS = {
    "bert": {},
    "medbert": dict(
        hidden_size=192,
        intermediate_size=64,
        num_attention_heads=6,
        num_hidden_layers=6,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    ),

    "cehrbert": dict(
        hidden_size=128,
        intermediate_size=2048,
        num_hidden_layers=12,
        num_attention_heads=8,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    ),

    "behrt": dict(
        hidden_size=288,
        intermediate_size=512,
        num_attention_heads=12,
        num_hidden_layers=6,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    ),

    "hibehrt": dict(
        hidden_size=150,
        intermediate_size=108,
        num_attention_heads=6,
        num_hidden_layers=4,
        hidden_dropout_prob=0.2,
        attention_probs_dropout_prob=0.3,
    ),
}



def get_config_and_model_cls(model_type: str, mode: str = "mlm", variant: str = None):
    assert mode in ["mlm", "eval", "causal"]

    if model_type not in CONFIG_MAPPING:
        raise ValueError(f"Unknown model_type: {model_type}")

    config_cls = CONFIG_MAPPING[model_type]

    if mode == "mlm":
        model_cls = MODEL_FOR_MASKED_LM_MAPPING[config_cls]
    elif mode == "eval":
        model_cls = MODEL_MAPPING[config_cls]
    else:
        model_cls = MODEL_FOR_CAUSAL_LM_MAPPING[config_cls]

    variant_kwargs = {}
    if variant is not None and issubclass(config_cls, BertConfig):
        variant_kwargs = BERT_VARIANTS.get(variant, {})
        if variant not in BERT_VARIANTS:
            raise ValueError(f"Unknown BERT variant: {variant}")

    def build_config(**kwargs):
        return config_cls(**variant_kwargs, **kwargs)

    return build_config, model_cls


def fix_roberta_longformer_max_pos(cfg):

    model_type = getattr(cfg, "model_type", "").lower()

    if model_type == "roberta":
        if cfg.max_position_embeddings == 512:
            cfg.max_position_embeddings = 513

    elif model_type == "longformer":
        if cfg.max_position_embeddings == 512:
            cfg.max_position_embeddings = 4097

    return cfg



def load_config_with_env(path):
    # read file
    with open(path, "r") as f:
        raw_text = f.read()
    expanded = os.path.expandvars(raw_text)
    
    return yaml.safe_load(expanded)


class Time2Vec(nn.Module):

    def __init__(
        self,
        in_features: int = 1,
        out_features: int = 16,
        periodic_activation: Callable = torch.sin,
    ):
        super().__init__()
        assert out_features >= 1, "out_features must be >= 1"

        self.in_features = in_features
        self.out_features = out_features
        self.periodic_activation = periodic_activation

        self.W = nn.Parameter(torch.randn(in_features, out_features - 1))
        self.b = nn.Parameter(torch.randn(out_features - 1))

        self.W0 = nn.Parameter(torch.randn(in_features))
        self.b0 = nn.Parameter(torch.randn(1))

    def forward(self, tau: torch.Tensor) -> torch.Tensor:

        v1 = self.periodic_activation(tau @ self.W + self.b)
        v2 = (tau @ self.W0).unsqueeze(-1) + self.b0

        return torch.cat([v2, v1], dim=-1)
    

def get_rank():
    if not dist.is_available() or not dist.is_initialized():
        return 0
    return dist.get_rank()




def get_bootstrap_ci(
    y_true: torch.Tensor,
    y_score: torch.Tensor,
    num_iter: int = 1000,
    alpha: float = 0.05,
    ndigits: int = 3,
):
    device = y_score.device

    y_true = y_true.detach().view(-1).to(device).long()
    y_score = y_score.detach().view(-1).to(device)

    auroc_point = BinaryAUROC().to(device)(y_score, y_true)
    auprc_point = BinaryAveragePrecision().to(device)(y_score, y_true)

    n = y_true.numel()
    auroc_samples = torch.empty(num_iter, device=device)
    auprc_samples = torch.empty(num_iter, device=device)

    for i in range(num_iter):
        idx = torch.randint(0, n, (n,), device=device)
        auroc_samples[i] = BinaryAUROC().to(device)(y_score[idx], y_true[idx])
        auprc_samples[i] = BinaryAveragePrecision().to(device)(y_score[idx], y_true[idx])

    # Percentile CI
    q_low = alpha / 2.0         # 2.5%
    q_high = 1.0 - alpha / 2.0  # 97.5%

    auroc_low = torch.quantile(auroc_samples, q_low)
    auroc_high = torch.quantile(auroc_samples, q_high)

    auprc_low = torch.quantile(auprc_samples, q_low)
    auprc_high = torch.quantile(auprc_samples, q_high)

    def _fmt(point, low, high):
        p = float(point.detach().cpu())
        l = float(low.detach().cpu())
        h = float(high.detach().cpu())
        return f"{round(p, ndigits)} ({round(l, ndigits)}, {round(h, ndigits)})"

    auroc_text = _fmt(auroc_point, auroc_low, auroc_high)
    auprc_text = _fmt(auprc_point, auprc_low, auprc_high)

    return auroc_text, auprc_text


def gather_1d_varlen_pl(module, x: torch.Tensor) -> torch.Tensor:
    x = x.detach().view(-1)

    if not getattr(module, "trainer", None) or module.trainer.world_size == 1:
        return x

    device = x.device
    local_len = torch.tensor([x.numel()], device=device, dtype=torch.long)

    all_lens = module.all_gather(local_len).view(-1) 
    max_len = int(all_lens.max().item())

    if x.numel() < max_len:
        pad = torch.zeros(max_len - x.numel(), device=device, dtype=x.dtype)
        x_pad = torch.cat([x, pad], dim=0)
    else:
        x_pad = x

    x_gather = module.all_gather(x_pad)

    chunks = []
    for r in range(x_gather.shape[0]):
        chunks.append(x_gather[r, : int(all_lens[r].item())])
    return torch.cat(chunks, dim=0)


def log_bootstrap_ci_text_percentile(
    module,
    y_true: torch.Tensor,
    y_score: torch.Tensor,
    prefix: str = "test",
    num_iter: int = 1000,
    alpha: float = 0.05,
    ndigits: int = 3,
):
    y_all = gather_1d_varlen_pl(module, y_true)
    s_all = gather_1d_varlen_pl(module, y_score)

    if not getattr(module, "trainer", None) or module.trainer.is_global_zero:
        auroc_ci_text, auprc_ci_text = get_bootstrap_ci(
            y_true=y_all,
            y_score=s_all,
            num_iter=num_iter,
            alpha=alpha,
            ndigits=ndigits,
        )

        # Use wandb.log directly for string-based CI values
#         wandb.log({
#             f"{prefix}_auroc_ci": auroc_ci_text,
#             f"{prefix}_auprc_ci": auprc_ci_text,
#         }, commit=False)
        print(f"{prefix}_auroc_ci = {auroc_ci_text}")
        print(f"{prefix}_auprc_ci = {auprc_ci_text}")

In [64]:
import torch
import torch.nn as nn
from typing import Any, Dict, Mapping


class CLMBREncoders(nn.Module):
    def __init__(
        self,
        model: nn.Module,
        batch_processor=None,
        normalize: bool = False,
        freeze: bool = False,
        device: str = None,
    ):
        super().__init__()

        self.model = model
        self.batch_processor = batch_processor
        self.normalize = normalize
        self.device_ = device or ("cuda" if torch.cuda.is_available() else "cpu")

        self.model.to(self.device_)

        if freeze:
            for p in self.model.parameters():
                p.requires_grad = False

    def forward(
        self,
        batch: Dict[str, Any],
        query_key: str = "query",
        history_key: str = "history",
    ) -> Dict[str, torch.Tensor]:

        q_batch = batch[query_key]
        h_batch = batch[history_key]

        q_vec = self._encode_packed_batch(q_batch)  # [B, H]

        h_vec_flat = self._encode_packed_batch(h_batch)  # [B*K, H]

        B, K = batch["history_valid_mask"].shape
        H = h_vec_flat.shape[-1]

        h_vec = h_vec_flat.reshape(B, K, H)

        return {
            "query_vec": q_vec,
            "hist_vec": h_vec,
        }

    def _encode_packed_batch(
        self,
        raw_batch: Mapping[str, Any],
    ) -> torch.Tensor:
#         if self.batch_processor is not None:
#             batch = self.batch_processor.collate([raw_batch])["batch"]
#             batch = femr.models.transformer.remove_first_dimension(batch)
#         else:
        batch = raw_batch

        transformer_batch = self._move_transformer_to_device(batch["transformer"])

        reprs = self.model.transformer(transformer_batch)

        patient_lengths = torch.as_tensor(
            transformer_batch["patient_lengths"],
            dtype=torch.long,
            device=reprs.device,
        )

        end_positions = torch.cumsum(patient_lengths, dim=0) - 1

        vec = reprs[end_positions]

        if self.normalize:
            vec = nn.functional.normalize(vec, p=2, dim=-1)

        return vec

    def _move_transformer_to_device(
        self,
        transformer_batch: Mapping[str, Any],
    ) -> Dict[str, Any]:

        out = {}

        for k, v in transformer_batch.items():
            if torch.is_tensor(v):
                out[k] = v.to(self.device_)
            else:
                out[k] = torch.as_tensor(v).to(self.device_)

        return out

In [65]:
encoder = CLMBREncoders(
    model=model,
    batch_processor=batch_processor,
    normalize=False,
    freeze=True,
)

In [66]:
class PrototypeRetrievalModule(nn.Module):
    def __init__(
        self,
        hidden_size: int = 768,
        num_prototypes: int = 256,
        query_temperature: float = 0.1,
        history_temperature: float = 0.025,
        normalize_prototypes: bool = True,
        softmax_threshold: float = 0.8,
        softmax_temperature: float = 1.0,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_prototypes = num_prototypes
        self.query_temperature = float(query_temperature)
        self.history_temperature = float(history_temperature)
        self.normalize_prototypes = normalize_prototypes
    
        self.softmax_threshold = float(softmax_threshold)
        self.softmax_temperature = float(softmax_temperature)

        self.prototypes = nn.Parameter(torch.empty(num_prototypes, hidden_size))
        nn.init.normal_(self.prototypes, mean=0.0, std=0.02)


    def _proto_probs(self, x: torch.Tensor, temperature: float) -> torch.Tensor:
        P = self.prototypes
        x = F.normalize(x, p=2, dim=-1)
        P = F.normalize(P, p=2, dim=-1)
        logits = x @ P.t()
        return F.softmax(logits / temperature, dim=-1)

    def _compute_neg_ce(self, q_probs: torch.Tensor, h_probs: torch.Tensor) -> torch.Tensor:
        return (q_probs.unsqueeze(1) * h_probs.clamp_min(1e-8).log()).sum(dim=-1)



    def _weights_and_mask_softmax(
        self,
        scores: torch.Tensor,
        valid_mask: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        if valid_mask is not None:
            valid_bool = valid_mask.bool()
            scores = scores.masked_fill(~valid_bool, -1e9)
        else:
            valid_bool = torch.ones_like(scores, dtype=torch.bool)

        weights = F.softmax(scores / self.softmax_temperature, dim=-1)

        hard = (weights >= self.softmax_threshold).float()
        keep = hard + weights - weights.detach()

        keep = keep * valid_bool.float()

        return {
            "attn_weights": weights,                  
            "attn_mask": keep,    
        }

    def _entropy(self, p: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
        p = p.clamp_min(eps)
        return -(p * p.log()).sum(dim=-1)

    def forward(
        self,
        query_vec: torch.Tensor,
        hist_vec: torch.Tensor,
        hist_valid_mask: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        B, K, _ = hist_vec.shape

        if hist_valid_mask is not None:
            hist_valid_mask = hist_valid_mask.to(device=hist_vec.device, dtype=torch.long)
            valid_f = hist_valid_mask.to(dtype=hist_vec.dtype)
            n_valid = valid_f.sum().clamp_min(1.0)
        else:
            valid_f = None
            n_valid = None

        q_probs = self._proto_probs(query_vec,temperature= self.query_temperature )   # [B, P]
        h_probs = self._proto_probs(hist_vec, temperature= self.history_temperature)    # [B, K, P]

        q_ent_all = self._entropy(q_probs)
        h_ent_all = self._entropy(h_probs)

        q_max = q_probs.max(dim=-1).values.mean()
        h_max_all = h_probs.max(dim=-1).values

        if valid_f is None:
            q_ent = q_ent_all.mean()
            h_ent = h_ent_all.mean()
            h_max = h_max_all.mean()
        else:
            q_ent = q_ent_all.mean()
            h_ent = (h_ent_all * valid_f).sum() / n_valid
            h_max = (h_max_all * valid_f).sum() / n_valid

        mean_q_probs = q_probs.mean(dim=0)
        if valid_f is None:
            mean_h_probs = h_probs.mean(dim=(0, 1))
        else:
            valid = valid_f.unsqueeze(-1)
            mean_h_probs = (h_probs * valid).sum(dim=(0, 1)) / valid.sum().clamp_min(1.0)

        mean_q_ent = self._entropy(mean_q_probs)
        mean_h_ent = self._entropy(mean_h_probs)

        neg_ce = self._compute_neg_ce(q_probs, h_probs)   # [B, K]

        wm = self._weights_and_mask_softmax(neg_ce, valid_mask=hist_valid_mask)

        out = {
            "neg_ce": neg_ce,
            "attn_mask": wm["attn_mask"],
            "attn_weights": wm["attn_weights"],
            "query_probs": q_probs,
            "hist_probs": h_probs,
            "diag_q_ent": q_ent,
            "diag_q_max": q_max,
            "diag_h_ent": h_ent,
            "diag_h_max": h_max,
            "diag_mean_q_ent": mean_q_ent,
            "diag_mean_h_ent": mean_h_ent,
        }
        return out

In [67]:
class FusionModule(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        num_layers: int = 2,
        num_heads: int = 4,
        ff_mult: int = 4,
        dropout: float = 0.1,
        use_weights_as_gating: bool = False,
        output_mode: str = "query",  
        return_seq: bool = False,
    ):
        super().__init__()
        self.use_weights_as_gating = use_weights_as_gating
        self.output_mode = output_mode
        self.return_seq = return_seq

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=ff_mult * hidden_size,
            dropout=dropout,
            activation="gelu",
            batch_first=True, 
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers,)

        self.out_norm = nn.LayerNorm(hidden_size)
        # self.hist_score_proj = nn.Linear(hidden_size, hidden_size)

    def forward(
        self,
        query_vec: torch.Tensor,
        hist_vec: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        attn_weights: Optional[torch.Tensor] = None,
        hist_valid_mask: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:

        B, K, H = hist_vec.shape
        assert query_vec.shape == (B, H)

        if self.use_weights_as_gating and attn_weights is not None:
            hist_vec = hist_vec * attn_weights.unsqueeze(-1).to(hist_vec.dtype)
            # hist_vec = self.hist_score_proj(hist_vec)

        # if self.use_weights_as_gating and attn_weights is not None:
        #     alpha = attn_weights.unsqueeze(-1).to(hist_vec.dtype)
        #     hist_vec = hist_vec * (1.0 + alpha)


        # if self.use_weights_as_gating and attn_weights is not None:
        #     mask = hist_valid_mask.float()                         # [B, K]
        #     num_valid = mask.sum(dim=1, keepdim=True).clamp_min(1.0)  # [B, 1]

        #     scaled_weights = attn_weights * num_valid              # rescale
        #     scaled_weights = scaled_weights * mask                 # zero out padding

        #     hist_vec = hist_vec * scaled_weights.unsqueeze(-1).to(hist_vec.dtype)

        elif attn_mask is not None:
            hist_vec = hist_vec * attn_mask.unsqueeze(-1).to(hist_vec.dtype)

        x = torch.cat([query_vec.unsqueeze(1), hist_vec], dim=1)  # [B, 1+K, H]

        if hist_valid_mask is None:
            src_key_padding_mask = None
            keep = torch.ones(B, 1 + K, device=x.device, dtype=x.dtype)
        else:
            query_valid = torch.ones(B, 1, device=x.device, dtype=torch.bool)
            keep_bool = torch.cat([query_valid, hist_valid_mask.bool()], dim=1)   # [B, 1+K]
            src_key_padding_mask = ~keep_bool
            keep = keep_bool.to(x.dtype)

        x_fused = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        x_fused = self.out_norm(x_fused)

        if self.output_mode == "query":
            fused_vec = x_fused[:, 0, :]
        elif self.output_mode == "mean":
            w = keep.unsqueeze(-1)
            denom = w.sum(dim=1).clamp_min(1.0)
            fused_vec = (x_fused * w).sum(dim=1) / denom

        out = {"fused_vec": fused_vec}
        if self.return_seq:
            out["fused_seq"] = x_fused
            out["fused_keep_mask"] = keep
        return out

In [68]:
class CLMBRRAPEvalModel(lt.LightningModule):
    def __init__(
        self,
        clmbr_model,
        batch_processor=None,
        hidden_size: int = 768,
        lr: float = 2e-5,
        wd: float = 0.001,
        max_epochs: int = 100,
        dropout: float = 0.1,
        freeze: bool = False,
        # --- prototype module ---
        num_prototypes: int = 256,
        query_temperature: float = 0.1,
        history_temperature: float = 0.025,
        normalize_prototypes: bool = True,
        softmax_threshold: float = 0.8,
        softmax_temperature: float = 1.0,
        sample_ent_lambda: float = 0.001,
        usage_ent_lambda: float = 0.001,
        use_prototypes: bool = False,
        # --- fusion module ---
        fusion_layers: int = 2,
        fusion_heads: int = 4,
        fusion_ff_mult: int = 4,
        fusion_output_mode: str = "query",
        use_weights_as_gating: bool = True,
    ):
        super().__init__()

        self.save_hyperparameters(ignore=["clmbr_model", "batch_processor"])

        self.hidden_size = hidden_size
        self.sample_ent_lambda = float(sample_ent_lambda)
        self.usage_ent_lambda = float(usage_ent_lambda)
        self.use_prototypes = use_prototypes

        self.encoders = CLMBREncoders(
            model=clmbr_model,
            batch_processor=batch_processor,
            normalize=False,
            freeze=freeze,
        )

        self.prototypes = PrototypeRetrievalModule(
            hidden_size=hidden_size,
            num_prototypes=num_prototypes,
            query_temperature=query_temperature,
            history_temperature=history_temperature,
            normalize_prototypes=normalize_prototypes,
            softmax_threshold=softmax_threshold,
            softmax_temperature=softmax_temperature,
        )

        self.fusion = FusionModule(
            hidden_size=hidden_size,
            num_layers=fusion_layers,
            num_heads=fusion_heads,
            ff_mult=fusion_ff_mult,
            dropout=dropout,
            use_weights_as_gating=use_weights_as_gating,
            output_mode=fusion_output_mode,
            return_seq=False,
        )

        self.classifier = nn.Linear(hidden_size, 1)
        self.criterion = nn.BCEWithLogitsLoss()

        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs

        self.train_auroc = BinaryAUROC()
        self.train_auprc = BinaryAveragePrecision()
        self.val_auroc = BinaryAUROC()
        self.val_auprc = BinaryAveragePrecision()
        self.test_auroc = BinaryAUROC()
        self.test_auprc = BinaryAveragePrecision()

        self._train_preds, self._train_labels = [], []
        self._val_preds, self._val_labels = [], []
        self._test_preds, self._test_labels = [], []

    def forward(self, batch: Dict[str, Any]) -> Dict[str, torch.Tensor]:
        enc_out = self.encoders(batch, query_key="query", history_key="history")

        proto_out = None
        attn_mask = None
        attn_weights = None
        hist_valid_mask = batch["history_valid_mask"]

        if self.use_prototypes:
            proto_out = self.prototypes(
                query_vec=enc_out["query_vec"],
                hist_vec=enc_out["hist_vec"],
                hist_valid_mask=hist_valid_mask,
            )
            attn_mask = proto_out["attn_mask"]
            attn_weights = proto_out["attn_weights"]

        fuse_out = self.fusion(
            query_vec=enc_out["query_vec"],
            hist_vec=enc_out["hist_vec"],
            attn_mask=attn_mask,
            attn_weights=attn_weights,
            hist_valid_mask=hist_valid_mask,
        )

        fused_vec = fuse_out["fused_vec"]
        logits = self.classifier(fused_vec).squeeze(-1)

        out = {
            "logits": logits,
            "fused_vec": fused_vec,
        }

        if proto_out is not None:
            out["proto"] = proto_out

        return out

    def shared_step(self, batch: Dict[str, Any], stage: str) -> torch.Tensor:
        out = self.forward(batch)
        proto = out.get("proto", None)

        if proto is not None and "diag_q_ent" in proto:
            self.log(
                f"{stage}_sample_q_ent",
                proto["diag_q_ent"],
                prog_bar=True,
                on_step=True,
                on_epoch=True,
                logger=True,
                sync_dist=True,
            )
            self.log(
                f"{stage}_q_max",
                proto["diag_q_max"],
                prog_bar=True,
                on_step=True,
                on_epoch=True,
                logger=True,
                sync_dist=True,
            )
            self.log(
                f"{stage}_sample_h_ent",
                proto["diag_h_ent"],
                prog_bar=True,
                on_step=True,
                on_epoch=True,
                logger=True,
                sync_dist=True,
            )
            self.log(
                f"{stage}_h_max",
                proto["diag_h_max"],
                prog_bar=True,
                on_step=True,
                on_epoch=True,
                logger=True,
                sync_dist=True,
            )

        if proto is not None and "diag_mean_q_ent" in proto:
            self.log(
                f"{stage}_batch_q_ent",
                proto["diag_mean_q_ent"],
                prog_bar=False,
                on_step=True,
                on_epoch=True,
                logger=True,
                sync_dist=True,
            )
            self.log(
                f"{stage}_batch_h_ent",
                proto["diag_mean_h_ent"],
                prog_bar=False,
                on_step=True,
                on_epoch=True,
                logger=True,
                sync_dist=True,
            )

        logits = out["logits"]
        y = batch["label"].float().view(-1)

        task_loss = self.criterion(logits, y)

        sample_ent_reg = task_loss.new_zeros(())
        usage_ent_reg = task_loss.new_zeros(())

        if proto is not None and "diag_q_ent" in proto and self.sample_ent_lambda > 0.0:
            sample_ent_reg = proto["diag_q_ent"] + proto["diag_h_ent"]

        if proto is not None and "diag_mean_q_ent" in proto and self.usage_ent_lambda > 0.0:
            usage_ent_reg = -(proto["diag_mean_q_ent"] + proto["diag_mean_h_ent"])

        total_loss = (
            task_loss
            + self.sample_ent_lambda * sample_ent_reg
            + self.usage_ent_lambda * usage_ent_reg
        )

        self.log(
            f"{stage}_loss",
            task_loss,
            prog_bar=False,
            on_step=True,
            on_epoch=True,
            logger=True,
            sync_dist=True,
        )
        self.log(
            f"{stage}_total_loss",
            total_loss,
            prog_bar=False,
            on_step=True,
            on_epoch=True,
            logger=True,
            sync_dist=True,
        )

        pos_score = torch.sigmoid(logits)

        if stage == "train":
            self._train_labels.append(y.detach())
            self._train_preds.append(pos_score.detach())
        elif stage == "val":
            self._val_labels.append(y.detach())
            self._val_preds.append(pos_score.detach())
        elif stage == "test":
            self._test_labels.append(y.detach())
            self._test_preds.append(pos_score.detach())

        if proto is not None and "attn_weights" in proto and proto["attn_weights"] is not None:
            with torch.no_grad():
                weights = proto["attn_weights"].float()
                mask = batch["history_valid_mask"].float()

                if not self.fusion.use_weights_as_gating:
                    hard_keep = (weights >= self.prototypes.softmax_threshold).float()
                    keep_rate = (hard_keep * mask).sum() / mask.sum().clamp_min(1.0)

                    self.log(
                        f"{stage}_keep_rate",
                        keep_rate,
                        prog_bar=False,
                        on_step=True,
                        on_epoch=True,
                        logger=True,
                        sync_dist=True,
                    )

                else:
                    weights = weights * mask
                    denom = weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
                    weights_norm = weights / denom

                    weight_entropy = -(
                        weights_norm * weights_norm.clamp_min(1e-8).log()
                    ).sum(dim=1)

                    valid_counts = mask.sum(dim=1).clamp_min(1.0)
                    max_entropy = valid_counts.log().clamp_min(1e-8)
                    weight_entropy_norm = (weight_entropy / max_entropy).mean()

                    self.log(
                        f"{stage}_weight_entropy",
                        weight_entropy.mean(),
                        prog_bar=False,
                        on_step=True,
                        on_epoch=True,
                        logger=True,
                        sync_dist=True,
                    )
                    self.log(
                        f"{stage}_weight_entropy_norm",
                        weight_entropy_norm,
                        prog_bar=False,
                        on_step=True,
                        on_epoch=True,
                        logger=True,
                        sync_dist=True,
                    )

        return total_loss

    def training_step(self, batch: Dict[str, Any], batch_idx: int) -> torch.Tensor:
        return self.shared_step(batch, stage="train")

    def validation_step(self, batch: Dict[str, Any], batch_idx: int) -> torch.Tensor:
        return self.shared_step(batch, stage="val")

    def test_step(self, batch: Dict[str, Any], batch_idx: int) -> torch.Tensor:
        return self.shared_step(batch, stage="test")

    def on_train_epoch_end(self) -> None:
        if not self._train_preds:
            return

        y = torch.cat(self._train_labels)
        p = torch.cat(self._train_preds)

        self.log("train_auroc",self.val_auroc(p, y.long()), on_epoch=True, logger=True, prog_bar=True, sync_dist=True,)
        self.log("train_auprc",self.val_auprc(p, y.long()),on_epoch=True, logger=True, prog_bar=True, sync_dist=True,)

        self._train_labels.clear()
        self._train_preds.clear()

    def on_validation_epoch_end(self) -> None:
        if not self._val_preds:
            return

        y = torch.cat(self._val_labels)
        p = torch.cat(self._val_preds)

        self.log("val_auroc",self.val_auroc(p, y.long()), on_epoch=True, logger=True, prog_bar=True, sync_dist=True,)
        self.log("val_auprc",self.val_auprc(p, y.long()),on_epoch=True, logger=True, prog_bar=True, sync_dist=True,)

        self._val_labels.clear()
        self._val_preds.clear()

    def on_test_epoch_end(self) -> None:
        if not self._test_preds:
            return

        y = torch.cat(self._test_labels)
        p = torch.cat(self._test_preds)

        self.log("test_auroc", self.test_auroc(p, y.long()), on_epoch=True, logger=True)
        self.log("test_auprc", self.test_auprc(p, y.long()), on_epoch=True, logger=True)

        log_bootstrap_ci_text_percentile(module=self,
                                         y_true=y,
                                         y_score=p,
                                         prefix="test",
                                         num_iter=1000,
                                         alpha=0.05,
                                         ndigits=3,)

        self._test_labels.clear()
        self._test_preds.clear()

    def configure_optimizers(self):
        param_groups = [{"params": self.encoders.parameters()},
                        {"params": self.fusion.parameters()},
                        {"params": self.classifier.parameters()}]

        if self.use_prototypes:
            param_groups.insert(1, {"params": self.prototypes.parameters()})

        optimizer = torch.optim.AdamW(param_groups,
                                    lr=self.lr,
#                                     momentum=0.9,
#                                     nesterov=True,
                                    betas=(0.9,0.95),
                                    weight_decay=self.wd)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer,
                                                               T_max=self.max_epochs,
                                                               eta_min=0.0)

        return {"optimizer": optimizer, "lr_scheduler": scheduler}

In [69]:
# for p in model.parameters():
#     p.requires_grad = True

In [70]:
rag_model = CLMBRRAPEvalModel(clmbr_model=model,
                                        batch_processor=None,
                                        hidden_size= 768,
                                        lr= 2e-5,
                                        wd=1e-2,
                                        max_epochs= 100,
                                        dropout=0.1,
                                        freeze=False,
                                        # --- prototype module ---
                                        num_prototypes=256,
                                        query_temperature= 0.025,
                                        history_temperature=0.1,
                                        normalize_prototypes= True,
                                        softmax_threshold= (1/24)*0.96,
                                        softmax_temperature=0.1,
                                        sample_ent_lambda=0,
                                        usage_ent_lambda= 0.004,
                                        use_prototypes= True,
                                        # --- fusion module ---
                                        fusion_layers= 2,
                                        fusion_heads= 4,
                                        fusion_ff_mult= 4,
                                        fusion_output_mode= 'query',
                                        use_weights_as_gating= True,
                                      )

from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

In [71]:
checkpoint_callback = ModelCheckpoint(#dirpath=ckpt_dir,
                                        monitor='val_auroc',
                                        mode='max',
                                        every_n_epochs=1,
                                        save_top_k=1)

early_stop = EarlyStopping(monitor='val_auroc',
                           min_delta=0.001,
                           mode='max', 
                           patience=4)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

In [72]:
trainer = lt.Trainer(accelerator='auto', 
                    devices='auto',
                    strategy='auto',
#                     logger=wandb_logger, 
                    log_every_n_steps=1,
                    num_sanity_val_steps=0,
                    max_epochs=100,
#                     precision='bf16-mixed',
                    callbacks=[early_stop,lr_monitor,checkpoint_callback],
                    )

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [73]:
# trainer.fit(model=rag_model,train_dataloaders=tloader,val_dataloaders=vloader)
# trainer.test(model=rag_model, dataloaders=teloader, ckpt_path='best')

In [85]:
import os

# ==========================================================
# Paths
# ==========================================================

data_idx_path = "clmbr_idx.parquet"
arrow_dataset_path = "ehrshot_clmbr_tokenized_arrow"

faiss_root = "./faiss_clmbr"

# ==========================================================
# Global settings
# ==========================================================

query_length = 512



tasks = ["chexpert"]

history_settings = [
    {
        "chunking_strategy": "overlap",
        "history_chunk_length": 256,
        "history_overlap": 32,
        "window_hours": 24.0,
    },
#     {
#         "chunking_strategy": "overlap",
#         "history_chunk_length": 512,
#         "history_overlap": 64,
#         "window_hours": 24.0,
#     },
#     {
#         "chunking_strategy": "time",
#         "history_chunk_length": 256,
#         "history_overlap": 0,
#         "window_hours": 24.0,
#     },
]

# ==========================================================
# Build indices
# ==========================================================

for task in tasks:
    for cfg in history_settings:

        if cfg["chunking_strategy"] == "time":
            span_dir = str(int(cfg["window_hours"]))
        else:
            span_dir = str(cfg["history_chunk_length"])

        save_path = os.path.join(
            faiss_root,
            str(query_length),
            cfg["chunking_strategy"],
            span_dir,
            task,
        )

        os.makedirs(save_path, exist_ok=True)

        print("=" * 80)
        print(
            f"Running CLMBR FAISS build | "
            f"task={task} | "
            f"strategy={cfg['chunking_strategy']} | "
            f"span={span_dir} | "
            f"query_length={query_length}"
        )
        print(f"save_path: {save_path}")
        print("=" * 80)

        build_clmbr_indices(
            data_idx_path=data_idx_path,
            arrow_dataset_path=arrow_dataset_path,
            save_path=save_path,
            model=model,
            batch_processor=batch_processor,
            task=task,
            query_length=query_length,
            history_chunk_length=cfg["history_chunk_length"],
            history_overlap=cfg["history_overlap"],
            chunking_strategy=cfg["chunking_strategy"],
            window_hours=cfg["window_hours"],
        )

Running CLMBR FAISS build | task=chexpert | strategy=overlap | span=256 | query_length=512
save_path: ./faiss_clmbr/512/overlap/256/chexpert


100%|██████████| 26275/26275 [40:33<00:00, 10.80it/s]  


example_id,subject_id,split,task,prediction_time,boolean_value,integer_value,float_value,categorical_value,arrow_row_idx,prediction_idx,history_min_idx,n_clmbr_tokens
u32,i64,str,str,datetime[μs],bool,i64,null,null,i64,i64,i64,i64
6995,115973501,"""held_out""","""lab_hyperkalemia""",2014-09-19 10:04:00,null,0,null,null,6406,28,0,1269
6996,115973501,"""held_out""","""lab_hyperkalemia""",2015-12-13 08:02:00,null,0,null,null,6406,330,0,1269
6997,115973501,"""held_out""","""lab_hyperkalemia""",2017-03-13 08:28:00,null,1,null,null,6406,447,0,1269
6998,115973501,"""held_out""","""lab_hyperkalemia""",2017-03-13 17:58:00,null,0,null,null,6406,510,0,1269
6999,115973501,"""held_out""","""lab_hyperkalemia""",2017-05-18 10:26:00,null,0,null,null,6406,699,0,1269
7000,115973501,"""held_out""","""lab_hyperkalemia""",2018-11-05 08:41:00,null,0,null,null,6406,1045,0,1269
7001,115973501,"""held_out""","""lab_hyperkalemia""",2019-11-11 08:16:00,null,0,null,null,6406,1213,0,1269
7002,115973469,"""held_out""","""lab_hyperkalemia""",1998-11-05 10:00:00,null,0,null,null,6374,19,0,4941
7003,115973469,"""held_out""","""lab_hyperkalemia""",1999-09-14 09:51:00,null,0,null,null,6374,41,0,4941


In [ ]:
"guo_readmission"
"new_hyperlipidemia"
"lab_thrombocytopenia"
"new_lupus"
"new_celiac"
"new_pancan"
"guo_los"
"new_acutemi"
"lab_hyponatremia"
"new_hypertension"
"chexpert"
"lab_hyperkalemia"
"lab_anemia"
"guo_icu"
"lab_hypoglycemia"